# TutorBot — Agente con herramientas + RAG

### Diplomado de desarrollo de aplicaciones con IA


### Intengrantes:
*   Eduardo Ruiz
*   Eduardo Velasquez
*   Diego Chiang
*   Robin Villegas
*   Mario Ampuero







## 1. Instalación

In [1]:
!pip install --quiet google-genai==1.56.0 fastmcp==3.1.1
!pip install --quiet langchain-community==0.4.1 langchain-text-splitters==1.1.1 faiss-cpu==1.13.2
!pip install --quiet transformers torch
!pip install --quiet -U gdown
!pip install --quiet gradio==6.24.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.6/426.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.2/236.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 34.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beh

## 2. Archivos del proyecto

El corpus incluye los materiales disponibles de todos los cursos y ciclos. Cada fragmento
conserva como metadata el ciclo, el curso, el tipo de contenido y el archivo de origen.


In [2]:
from pathlib import Path
import shutil
import requests
import json

ZIP_URL = (
    "https://github.com/DiegoChiang/TutorBot/"
    "releases/latest/download/Utils_Chatbot.zip"
)

ZIP_PATH = Path("/content/Utils_Chatbot.zip")
EXTRACT_DIR = Path("/content/Utils_Chatbot_extraido")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Descargando Utils_Chatbot.zip desde GitHub Releases...")

with requests.get(ZIP_URL, stream=True, timeout=300) as response:
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    descargado = 0

    with open(ZIP_PATH, "wb") as f:
        for bloque in response.iter_content(chunk_size=1024 * 1024):
            if not bloque:
                continue

            f.write(bloque)
            descargado += len(bloque)

            if total:
                print(
                    f"\rDescargando: {descargado / total * 100:.1f}%",
                    end=""
                )

print(
    f"\nZIP descargado: "
    f"{ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB"
)

shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_DIR))

candidatos = [EXTRACT_DIR]
candidatos.extend(
    p for p in EXTRACT_DIR.iterdir()
    if p.is_dir()
)

PROJECT_DIR = next(
    (
        p for p in candidatos
        if (p / "corpus").is_dir()
        and (p / "codigo").is_dir()
    ),
    None
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró una carpeta con 'corpus' y 'codigo'."
    )

CORPUS_DIR = PROJECT_DIR / "corpus"
CODE_DIR = PROJECT_DIR / "codigo"
DATA_DIR = PROJECT_DIR / "datos"
RESULTS_DIR = PROJECT_DIR / "resultados"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_PATH = DATA_DIR / "chunks.json"
EVAL_SET_PATH = DATA_DIR / "eval_set.json"

print("\nCarga completada.")
print("Proyecto:", PROJECT_DIR)
print("Corpus:", CORPUS_DIR)
print("Código:", CODE_DIR)
print("Datos:", DATA_DIR)
print("Resultados:", RESULTS_DIR)


Descargando Utils_Chatbot.zip desde GitHub Releases...
Descargando: 100.0%
ZIP descargado: 86.9 MB

Carga completada.
Proyecto: /content/Utils_Chatbot_extraido/Utils_Chatbot
Corpus: /content/Utils_Chatbot_extraido/Utils_Chatbot/corpus
Código: /content/Utils_Chatbot_extraido/Utils_Chatbot/codigo
Datos: /content/Utils_Chatbot_extraido/Utils_Chatbot/datos
Resultados: /content/Utils_Chatbot_extraido/Utils_Chatbot/resultados


In [3]:
import re
import unicodedata
from collections import Counter, defaultdict

def decodificar_hash_unicode(texto):
    return re.sub(
        r"#U([0-9A-Fa-f]{4})",
        lambda m: chr(int(m.group(1), 16)),
        str(texto)
    )

def normalizar_nombre(texto):
    texto = decodificar_hash_unicode(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )
    texto = re.sub(r"[^a-z0-9]+", " ", texto.lower())
    return re.sub(r"\s+", " ", texto).strip()

REGLAS_CURSOS = [
    (
        "Fundamentos de Machine Learning",
        ["fundamentos de machine learning"]
    ),
    (
        "Python para Ciencia de Datos",
        ["python para ciencia de datos", "python para ciencias de datos"]
    ),
    (
        "Visualización de Datos",
        ["visualizacion de datos"]
    ),
    (
        "Análisis de Sentimientos",
        ["analisis de sentimiento", "analisis de sentimientos"]
    ),
    (
        "Desarrollo de Aplicaciones con Visión Artificial",
        ["desarrollo en aplicaciones con vision artificial"]
    ),
    (
        "Inteligencia Artificial para Juegos",
        ["inteligencia artificial para juegos"]
    ),
    (
        "Diseño de Chatbots Conversacional",
        ["diseno de chatbot"]
    ),
    (
        "Optimización Industrial con Computación Evolutiva",
        ["optimizacion industrial con computacion evolutiva"]
    ),
    (
        "Redes Neuronales para el Análisis de Series Temporales",
        [
            "redes neuronales para el analisis de series temporales",
            "redes neuronales para series temporales"
        ]
    ),
]

def canonicalizar_curso(nombre):
    nombre_norm = normalizar_nombre(nombre)

    for canonico, variantes in REGLAS_CURSOS:
        if any(
            normalizar_nombre(v) in nombre_norm
            for v in variantes
        ):
            return canonico

    return decodificar_hash_unicode(nombre)

def extraer_ciclo(partes):
    for parte in partes:
        m = re.match(
            r"Ciclo\s+(\d+)",
            decodificar_hash_unicode(parte),
            flags=re.IGNORECASE
        )
        if m:
            return int(m.group(1))
    return None

def metadata_ruta(path):
    relativa = path.relative_to(CORPUS_DIR)
    partes = list(relativa.parts)

    ciclo = extraer_ciclo(partes)
    curso_raw = (
        partes[2]
        if len(partes) >= 4
        else "Curso no identificado"
    )

    return {
        "ciclo": ciclo,
        "curso": canonicalizar_curso(curso_raw),
        "curso_original": decodificar_hash_unicode(curso_raw),
        "ruta_relativa": decodificar_hash_unicode(str(relativa)),
    }

inventario = []

for path in CORPUS_DIR.rglob("*"):
    if not path.is_file():
        continue
    if path.suffix.lower() not in {".md", ".txt", ".ipynb"}:
        continue

    meta = metadata_ruta(path)
    inventario.append(
        (
            meta["ciclo"],
            meta["curso"],
            path.suffix.lower()
        )
    )

print("Archivos de corpus:", len(inventario))
print("\nCursos detectados:")

conteo = Counter((ciclo, curso) for ciclo, curso, _ in inventario)

for (ciclo, curso), cantidad in sorted(
    conteo.items(),
    key=lambda x: (x[0][0] or 99, x[0][1])
):
    print(f"  Ciclo {ciclo} | {curso}: {cantidad} archivos")


Archivos de corpus: 235

Cursos detectados:
  Ciclo 1 | Fundamentos de Machine Learning: 24 archivos
  Ciclo 1 | Python para Ciencia de Datos: 22 archivos
  Ciclo 1 | Visualización de Datos: 26 archivos
  Ciclo 2 | Análisis de Sentimientos: 30 archivos
  Ciclo 2 | Desarrollo de Aplicaciones con Visión Artificial: 30 archivos
  Ciclo 2 | Inteligencia Artificial para Juegos: 31 archivos
  Ciclo 3 | Diseño de Chatbots Conversacional: 29 archivos
  Ciclo 3 | Optimización Industrial con Computación Evolutiva: 15 archivos
  Ciclo 3 | Redes Neuronales para el Análisis de Series Temporales: 28 archivos


### Generación de `chunks.json` y `eval_set.json`

La ingesta procesa `Markdown Textos`, `Markdown Videos` y `Notebooks` de todos los cursos.


In [4]:
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n\n", "\n\n", "\n", " ", ","],
)

def extraer_clase(nombre):
    texto = normalizar_nombre(nombre)

    for patron in [
        r"\bclase\s*0*(\d+)\b",
        r"\bsesion\s*0*(\d+)\b",
        r"\bs\s*0*(\d+)\b",
        r"\bnb\s*0*(\d+)\b",
    ]:
        m = re.search(patron, texto)
        if m:
            return int(m.group(1))

    return None

def limpiar_texto(texto):
    texto = texto.replace("\r\n", "\n")
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

TS_INLINE_RE = re.compile(
    r"^\[(?:(\d+):)?(\d+):(\d{2})\]\s*(.*)$"
)
TS_SOLO_RE = re.compile(
    r"^(?:(\d+):)?(\d+):(\d{2})\s*$"
)

def formatear_ts(grupos):
    h, mm, ss = grupos
    if h is None:
        return f"{int(mm)}:{ss}"
    return f"{int(h)}:{int(mm):02d}:{ss}"

def parsear_transcripcion(texto):
    partes = []
    marcas = []
    offset = 0
    pendiente = None

    for linea in texto.splitlines():
        linea = linea.strip()

        if not linea:
            continue

        inline = TS_INLINE_RE.match(linea)

        if inline:
            ts = formatear_ts(inline.groups()[:3])
            contenido = inline.group(4).strip()

            if contenido:
                marcas.append((offset, ts))
                partes.append(contenido)
                offset += len(contenido) + 1
            continue

        solo = TS_SOLO_RE.match(linea)

        if solo:
            pendiente = formatear_ts(solo.groups())
            continue

        if pendiente:
            marcas.append((offset, pendiente))
            pendiente = None

        partes.append(linea)
        offset += len(linea) + 1

    return " ".join(partes), marcas

def timestamp_para(offset, marcas):
    actual = marcas[0][1] if marcas else None

    for posicion, ts in marcas:
        if posicion > offset:
            break
        actual = ts

    return actual

def source_celda(celda):
    source = celda.get("source", "")
    if isinstance(source, list):
        return "".join(source).strip()
    return str(source).strip()

PALABRAS_ADMIN = {
    "rubrica",
    "trabajo",
    "tarea",
    "syllabus",
    "indicaciones",
    "entregable",
}

def tipo_markdown(path):
    nombre = normalizar_nombre(path.name)
    return (
        "administrativo"
        if any(p in nombre for p in PALABRAS_ADMIN)
        else "slide"
    )

chunks = []

def agregar_chunks_texto(texto, metadata, marcas=None):
    texto = limpiar_texto(texto)
    if not texto:
        return

    cursor = 0
    contador = 0

    for parte in splitter.split_text(texto):
        parte = parte.strip()

        if len(parte) < 80:
            continue

        meta = dict(metadata)

        if marcas:
            posicion = texto.find(parte, cursor)
            if posicion == -1:
                posicion = cursor
            cursor = posicion + 1
            meta["timestamp"] = timestamp_para(posicion, marcas)

        meta["chunk_id"] = (
            f"{meta['ruta_relativa']}#{contador:03d}"
        )
        contador += 1

        chunks.append(
            {
                "id": len(chunks),
                "texto": parte.replace("\n", " "),
                "metadata": meta,
            }
        )

# Markdown Textos
raiz_textos = CORPUS_DIR / "Markdown Textos"

for path in sorted(raiz_textos.rglob("*.md")):
    meta = metadata_ruta(path)
    nombre = decodificar_hash_unicode(path.name)

    agregar_chunks_texto(
        path.read_text(encoding="utf-8", errors="ignore"),
        {
            **meta,
            "clase": extraer_clase(nombre),
            "titulo_clase": re.sub(
                r"\.md$",
                "",
                nombre,
                flags=re.IGNORECASE
            ),
            "tipo": tipo_markdown(path),
            "fuente": nombre,
        }
    )

# Markdown Videos
raiz_videos = CORPUS_DIR / "Markdown Videos"

for path in sorted(raiz_videos.rglob("*.txt")):
    meta = metadata_ruta(path)
    nombre = decodificar_hash_unicode(path.name)

    texto, marcas = parsear_transcripcion(
        path.read_text(encoding="utf-8", errors="ignore")
    )

    agregar_chunks_texto(
        texto,
        {
            **meta,
            "clase": extraer_clase(nombre),
            "titulo_clase": re.sub(
                r"\.txt$",
                "",
                nombre,
                flags=re.IGNORECASE
            ),
            "tipo": "transcripcion",
            "fuente": nombre,
        },
        marcas=marcas
    )

# Notebooks
raiz_notebooks = CORPUS_DIR / "Notebooks"

for path in sorted(raiz_notebooks.rglob("*.ipynb")):
    meta = metadata_ruta(path)
    nombre = decodificar_hash_unicode(path.name)
    clase = extraer_clase(nombre)

    titulo = re.sub(
        r"_no_outputs\.ipynb$",
        "",
        nombre,
        flags=re.IGNORECASE
    )
    titulo = re.sub(
        r"\.ipynb$",
        "",
        titulo,
        flags=re.IGNORECASE
    )

    with path.open(encoding="utf-8") as f:
        notebook = json.load(f)

    seccion = ""

    for i, celda in enumerate(notebook.get("cells", [])):
        fuente = source_celda(celda)

        if not fuente:
            continue

        if celda.get("cell_type") == "markdown":
            titulos = [
                linea.lstrip("#").strip()
                for linea in fuente.splitlines()
                if linea.lstrip().startswith("#")
            ]
            if titulos:
                seccion = titulos[0]
            continue

        if celda.get("cell_type") != "code":
            continue

        encabezado = (
            f"[Ciclo {meta['ciclo']} · "
            f"{meta['curso']} · {titulo}"
        )
        if seccion:
            encabezado += f" · {seccion}"
        encabezado += "]\n"

        partes = (
            [fuente]
            if len(fuente) <= 1500
            else splitter.split_text(fuente)
        )

        for j, parte in enumerate(partes):
            m = {
                **meta,
                "clase": clase,
                "titulo_clase": titulo,
                "tipo": "codigo",
                "fuente": nombre,
                "seccion": seccion,
                "celda": i,
            }

            sufijo = (
                f"#c{i:03d}"
                if len(partes) == 1
                else f"#c{i:03d}-{j}"
            )

            m["chunk_id"] = (
                f"{meta['ruta_relativa']}{sufijo}"
            )

            chunks.append(
                {
                    "id": len(chunks),
                    "texto": encabezado + parte,
                    "metadata": m,
                }
            )

chunk_ids = [
    c["metadata"]["chunk_id"]
    for c in chunks
]

duplicados = [
    cid
    for cid, n in Counter(chunk_ids).items()
    if n > 1
]

if duplicados:
    raise ValueError(
        "Se generaron chunk_id duplicados: "
        + ", ".join(duplicados[:10])
    )

CHUNKS_PATH.write_text(
    json.dumps(
        chunks,
        ensure_ascii=False,
        indent=1
    ),
    encoding="utf-8"
)

print("chunks.json:", CHUNKS_PATH)
print("Total de chunks:", len(chunks))

print("\nPor tipo:")
for tipo, cantidad in Counter(
    c["metadata"]["tipo"]
    for c in chunks
).most_common():
    print(f"  {tipo}: {cantidad}")

print("\nPor ciclo y curso:")
conteo = Counter(
    (
        c["metadata"]["ciclo"],
        c["metadata"]["curso"]
    )
    for c in chunks
)

for (ciclo, curso), cantidad in sorted(
    conteo.items(),
    key=lambda x: (x[0][0] or 99, x[0][1])
):
    print(
        f"  Ciclo {ciclo} | "
        f"{curso}: {cantidad}"
    )


chunks.json: /content/Utils_Chatbot_extraido/Utils_Chatbot/datos/chunks.json
Total de chunks: 15412

Por tipo:
  transcripcion: 8210
  slide: 3779
  codigo: 3402
  administrativo: 21

Por ciclo y curso:
  Ciclo 1 | Fundamentos de Machine Learning: 2200
  Ciclo 1 | Python para Ciencia de Datos: 1303
  Ciclo 1 | Visualización de Datos: 1199
  Ciclo 2 | Análisis de Sentimientos: 1718
  Ciclo 2 | Desarrollo de Aplicaciones con Visión Artificial: 1751
  Ciclo 2 | Inteligencia Artificial para Juegos: 2397
  Ciclo 3 | Diseño de Chatbots Conversacional: 2332
  Ciclo 3 | Optimización Industrial con Computación Evolutiva: 972
  Ciclo 3 | Redes Neuronales para el Análisis de Series Temporales: 1540


In [5]:
def score_chunk(chunk, keywords):
    texto = normalizar_nombre(chunk["texto"])
    return sum(
        texto.count(normalizar_nombre(k))
        for k in keywords
        if normalizar_nombre(k)
    )

def referencias_para(curso, tipos, keywords, max_refs=3):
    candidatos = [
        c for c in chunks
        if c["metadata"].get("curso") == curso
        and c["metadata"].get("tipo") in tipos
    ]

    if not candidatos:
        return []

    ordenados = sorted(
        candidatos,
        key=lambda c: (
            score_chunk(c, keywords),
            len(c["texto"])
        ),
        reverse=True
    )

    positivos = [
        c for c in ordenados
        if score_chunk(c, keywords) > 0
    ]

    seleccion = (
        positivos[:max_refs]
        if positivos
        else ordenados[:max_refs]
    )

    return [
        {
            "chunk_id": c["metadata"]["chunk_id"],
            "tipo": c["metadata"]["tipo"],
            "fuente": c["metadata"]["fuente"],
        }
        for c in seleccion
    ]

C = {
    "ml": "Fundamentos de Machine Learning",
    "python": "Python para Ciencia de Datos",
    "viz": "Visualización de Datos",
    "sent": "Análisis de Sentimientos",
    "vision": "Desarrollo de Aplicaciones con Visión Artificial",
    "games": "Inteligencia Artificial para Juegos",
    "chat": "Diseño de Chatbots Conversacional",
    "opt": "Optimización Industrial con Computación Evolutiva",
    "series": "Redes Neuronales para el Análisis de Series Temporales",
}

CONCEPTOS = [
    (
        C["ml"],
        "¿Cuál es la diferencia entre aprendizaje supervisado y no supervisado?",
        ["aprendizaje supervisado", "aprendizaje no supervisado", "etiquetas"],
        "En aprendizaje supervisado se aprende con una variable objetivo o etiqueta; "
        "en el no supervisado se buscan patrones sin una etiqueta objetivo."
    ),
    (
        C["python"],
        "¿Para qué se utiliza Pandas al trabajar con datos tabulares?",
        ["pandas", "dataframe", "datos tabulares"],
        "Pandas permite cargar, organizar, transformar y analizar datos tabulares "
        "mediante estructuras como DataFrame."
    ),
    (
        C["viz"],
        "¿Cuál es el propósito de una visualización de datos?",
        ["visualizacion", "datos", "grafico"],
        "Una visualización representa datos gráficamente para facilitar su exploración, "
        "comparación, interpretación y comunicación."
    ),
    (
        C["sent"],
        "¿Por qué se realiza preprocesamiento de textos antes de analizarlos?",
        ["preprocesamiento", "texto", "token"],
        "El preprocesamiento limpia y transforma el texto para obtener una representación "
        "más adecuada para el análisis o modelado posterior."
    ),
    (
        C["vision"],
        "¿Qué es una CNN y por qué se usa en visión artificial?",
        ["CNN", "convolucional", "imagen"],
        "Una CNN aprende características espaciales de las imágenes mediante operaciones "
        "convolucionales y se usa ampliamente en tareas de visión artificial."
    ),
    (
        C["games"],
        "¿Qué relación existe entre un agente y su entorno?",
        ["agente", "entorno", "accion", "percepcion"],
        "Un agente percibe información del entorno y ejecuta acciones sobre él según "
        "sus objetivos o su política de decisión."
    ),
    (
        C["chat"],
        "¿Qué es un intent dentro de NLU?",
        ["intent", "intencion", "NLU"],
        "Un intent representa la intención o propósito asociado al mensaje del usuario."
    ),
    (
        C["opt"],
        "¿Qué es un algoritmo genético?",
        ["algoritmo genetico", "poblacion", "seleccion", "mutacion"],
        "Un algoritmo genético trabaja con una población de soluciones y aplica selección "
        "y operadores de variación para buscar mejores individuos."
    ),
    (
        C["series"],
        "¿Qué caracteriza a una serie temporal?",
        ["serie temporal", "tiempo", "observaciones"],
        "Una serie temporal es una secuencia de observaciones ordenadas en el tiempo."
    ),
]

CODIGO = [
    (
        C["ml"],
        "¿Qué hace print_confusion_matrix en el notebook de Naive Bayes?",
        ["print_confusion_matrix"],
        "La función muestra la matriz de confusión para revisar aciertos y errores "
        "de clasificación."
    ),
    (
        C["python"],
        "¿Qué hace la función es_par del material de Python?",
        ["def es_par", "es_par"],
        "La función determina si un número es par comprobando su divisibilidad entre dos."
    ),
    (
        C["viz"],
        "¿Cómo se crea un gráfico de barras con Matplotlib en los notebooks del curso?",
        ["plt.bar", "bar"],
        "El gráfico de barras se construye con la función bar de Matplotlib."
    ),
    (
        C["sent"],
        "¿Qué hace preprocess_text en el notebook de modelado de tópicos?",
        ["preprocess_text"],
        "preprocess_text prepara el contenido textual antes de utilizarlo en el modelado "
        "de tópicos."
    ),
    (
        C["vision"],
        "¿Qué hace la función convolucion del notebook de filtrado?",
        ["def convolucion", "convolucion"],
        "La función aplica una operación de convolución entre una imagen y un kernel o filtro."
    ),
    (
        C["games"],
        "¿Qué representa la clase QLearningAgent?",
        ["class QLearningAgent", "QLearningAgent"],
        "QLearningAgent implementa un agente que aprende valores Q a partir de la experiencia."
    ),
    (
        C["chat"],
        "¿Qué hace la función predict_intent?",
        ["predict_intent"],
        "predict_intent procesa una entrada y obtiene la intención predicha por el modelo."
    ),
    (
        C["opt"],
        "¿Qué hace la función PSO del notebook PSO_ABC?",
        ["def PSO", "PSO_particles", "Particle"],
        "La función ejecuta Particle Swarm Optimization actualizando una población "
        "de partículas para buscar mejores soluciones."
    ),
    (
        C["series"],
        "¿Qué hace get_QRS_target en los notebooks de ECG?",
        ["get_QRS_target"],
        "get_QRS_target construye el objetivo utilizado para identificar o segmentar "
        "los complejos QRS."
    ),
]

TAREAS = [
    (
        C["ml"],
        "Quiero entrenar un modelo de regresión logística",
        ["regresion logistica", "logistic", "train"],
        "La guía debe preparar variables y etiquetas, separar datos, ajustar el modelo "
        "y evaluar sus predicciones."
    ),
    (
        C["python"],
        "Quiero crear un dashboard con Streamlit",
        ["streamlit", "dashboard"],
        "La guía debe preparar los datos, construir la interfaz y visualizaciones "
        "en Streamlit y ejecutar la aplicación."
    ),
    (
        C["viz"],
        "Quiero visualizar una serie temporal con Plotly",
        ["plotly", "datos temporales", "time"],
        "La guía debe preparar la variable temporal y construir una visualización "
        "interactiva con Plotly."
    ),
    (
        C["sent"],
        "Quiero construir un clasificador de sentimientos",
        ["analisis sentimientos", "preprocesamiento", "representacion textos"],
        "La guía debe preprocesar los textos, representarlos, entrenar el modelo y "
        "evaluar la clasificación."
    ),
    (
        C["vision"],
        "Quiero entrenar una CNN para clasificar imágenes",
        ["CNN", "convolucional", "train", "clasificacion"],
        "La guía debe preparar las imágenes, definir la CNN, entrenarla y evaluar "
        "su desempeño."
    ),
    (
        C["games"],
        "Quiero implementar un agente con Q-learning",
        ["Q-learning", "QLearningAgent", "qvalues", "recompensa"],
        "La guía debe definir estados, acciones y recompensas, actualizar valores Q "
        "y derivar una política."
    ),
    (
        C["chat"],
        "Quiero implementar RAG en mi chatbot",
        ["RAG", "embedding", "retrieval", "FAISS"],
        "La guía debe dividir documentos en chunks, crear embeddings, indexarlos, "
        "recuperar contexto y generar la respuesta."
    ),
    (
        C["opt"],
        "Quiero resolver un problema de optimización continua con PSO",
        ["PSO", "Particle", "ackley", "rastrigin"],
        "La guía debe definir la función objetivo, inicializar partículas, actualizar "
        "posición y velocidad y conservar las mejores soluciones."
    ),
    (
        C["series"],
        "Quiero segmentar complejos QRS usando una LSTM",
        ["QRS", "LSTM", "segmentacion", "ECG"],
        "La guía debe preparar la señal y etiquetas QRS, crear secuencias, entrenar "
        "la LSTM y evaluar la segmentación."
    ),
]

eval_set = []
contador = 1

for curso, pregunta, keywords, respuesta in CONCEPTOS:
    eval_set.append(
        {
            "id": f"EV{contador:02d}",
            "curso": curso,
            "categoria": "conceptual",
            "pregunta": pregunta,
            "herramienta_esperada": "responder_concepto",
            "respuesta_correcta": respuesta,
            "chunks_relevantes": referencias_para(
                curso,
                {"slide", "transcripcion"},
                keywords
            ),
        }
    )
    contador += 1

for curso, pregunta, keywords, respuesta in CODIGO:
    eval_set.append(
        {
            "id": f"EV{contador:02d}",
            "curso": curso,
            "categoria": "codigo",
            "pregunta": pregunta,
            "herramienta_esperada": "explicar_codigo",
            "respuesta_correcta": respuesta,
            "chunks_relevantes": referencias_para(
                curso,
                {"codigo"},
                keywords
            ),
        }
    )
    contador += 1

for curso, pregunta, keywords, respuesta in TAREAS:
    eval_set.append(
        {
            "id": f"EV{contador:02d}",
            "curso": curso,
            "categoria": "tarea",
            "pregunta": pregunta,
            "herramienta_esperada": "guiar_tarea",
            "respuesta_correcta": respuesta,
            "chunks_relevantes": referencias_para(
                curso,
                {"slide", "transcripcion", "codigo", "administrativo"},
                keywords
            ),
        }
    )
    contador += 1

EXTRAS = [
    (
        "ambiguo",
        "¿Cómo lo hago?",
        "pedir_aclaracion",
        "TutorBot debe pedir información adicional para saber a qué tarea o curso se refiere."
    ),
    (
        "ambiguo",
        "No me funciona",
        "pedir_aclaracion",
        "TutorBot debe pedir que se especifique qué componente, código o actividad está fallando."
    ),
    (
        "ambiguo",
        "Explícame el modelo",
        "pedir_aclaracion",
        "TutorBot debe pedir que se indique qué modelo o curso se desea consultar."
    ),
    (
        "fuera_de_alcance",
        "¿Quién ganó la Copa del Mundo de 2022?",
        "responder_concepto",
        "La herramienta conceptual debe activar el fallback."
    ),
    (
        "fuera_de_alcance",
        "¿Cuál es la capital de Japón?",
        "responder_concepto",
        "La herramienta conceptual debe activar el fallback."
    ),
]

for categoria, pregunta, herramienta, respuesta in EXTRAS:
    eval_set.append(
        {
            "id": f"EV{contador:02d}",
            "curso": None,
            "categoria": categoria,
            "pregunta": pregunta,
            "herramienta_esperada": herramienta,
            "respuesta_correcta": respuesta,
            "chunks_relevantes": [],
        }
    )
    contador += 1

sin_refs = [
    c["id"]
    for c in eval_set
    if c["categoria"] in {"conceptual", "codigo", "tarea"}
    and not c["chunks_relevantes"]
]

if sin_refs:
    raise ValueError(
        "Casos sin referencias: "
        + ", ".join(sin_refs)
    )

EVAL_SET_PATH.write_text(
    json.dumps(
        eval_set,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("eval_set.json:", EVAL_SET_PATH)
print("Casos:", len(eval_set))
print(
    "Con referencias:",
    sum(bool(c["chunks_relevantes"]) for c in eval_set)
)

print("\nCasos evaluados por curso:")
for curso, cantidad in sorted(
    Counter(
        c["curso"]
        for c in eval_set
        if c["curso"]
    ).items()
):
    print(f"  {curso}: {cantidad}")


eval_set.json: /content/Utils_Chatbot_extraido/Utils_Chatbot/datos/eval_set.json
Casos: 32
Con referencias: 27

Casos evaluados por curso:
  Análisis de Sentimientos: 3
  Desarrollo de Aplicaciones con Visión Artificial: 3
  Diseño de Chatbots Conversacional: 3
  Fundamentos de Machine Learning: 3
  Inteligencia Artificial para Juegos: 3
  Optimización Industrial con Computación Evolutiva: 3
  Python para Ciencia de Datos: 3
  Redes Neuronales para el Análisis de Series Temporales: 3
  Visualización de Datos: 3


In [6]:
from collections import Counter

with open(
    CHUNKS_PATH,
    encoding="utf-8"
) as f:
    chunks = json.load(
        f
    )

with open(
    EVAL_SET_PATH,
    encoding="utf-8"
) as f:
    eval_set = json.load(
        f
    )


print(
    f"Chunks: {len(chunks)} | "
    f"Casos de evaluación: {len(eval_set)}"
)

print(
    "Tipos:",
    Counter(
        c[
            "metadata"
        ][
            "tipo"
        ]
        for c in chunks
    )
)

print(
    "Por clase:",
    Counter(
        c[
            "metadata"
        ].get(
            "clase"
        )
        for c in chunks
    )
)


chunk_ids_corpus = {
    c[
        "metadata"
    ].get(
        "chunk_id"
    )
    for c in chunks
    if c.get(
        "metadata",
        {}
    ).get(
        "chunk_id"
    )
}


referencias_faltantes = []

for caso in eval_set:
    for ref in caso.get(
        "chunks_relevantes",
        []
    ):
        cid = ref.get(
            "chunk_id"
        )

        if cid not in chunk_ids_corpus:
            referencias_faltantes.append(
                (
                    caso.get(
                        "id"
                    ),
                    cid
                )
            )


if referencias_faltantes:
    print(
        "\nReferencias que no existen "
        "en chunks.json:"
    )

    for caso_id, cid in (
        referencias_faltantes[
            :30
        ]
    ):
        print(
            f"  {caso_id}: "
            f"{cid}"
        )

    raise ValueError(
        f"Hay "
        f"{len(referencias_faltantes)} "
        "referencias desincronizadas."
    )


print(
    "Validación corpus/eval_set: "
    "SIN ERRORES."
)


Chunks: 15412 | Casos de evaluación: 32
Tipos: Counter({'transcripcion': 8210, 'slide': 3779, 'codigo': 3402, 'administrativo': 21})
Por clase: Counter({None: 5947, 2: 1462, 4: 1378, 5: 1354, 3: 1330, 1: 1305, 6: 1178, 7: 925, 2026: 289, 8: 191, 9: 22, 11: 16, 12: 9, 10: 6})
Validación corpus/eval_set: SIN ERRORES.


## 3. Configuración de Gemini

Se utilizan modelos separados según su función:

- `MODEL_CHAT`: genera las respuestas finales de TutorBot.
- `MODEL_ROUTING`: selecciona la herramienta adecuada.
- `MODEL_JUEZ`: evalúa las respuestas generadas.

La API key se obtiene desde un Secret de Google Colab llamado `GEMINI_API_KEY`.


In [7]:
from google import genai
from google.genai.types import GenerateContentConfig
from google.colab import userdata

# Modelo principal para las respuestas finales
MODEL_CHAT = "gemini-3.1-pro-preview"

# Modelo para seleccionar herramientas
MODEL_ROUTING = "gemini-3.1-pro-preview"

# Modelo independiente para evaluar las respuestas
MODEL_JUEZ = "gemini-3.6-flash"

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "No se encontró GEMINI_API_KEY en los Secrets de Google Colab."
    )

gemini_client = genai.Client(api_key=api_key)

print("Gemini configurado.")
print("Modelo de TutorBot:", MODEL_CHAT)
print("Modelo de routing:", MODEL_ROUTING)
print("Modelo juez:", MODEL_JUEZ)


Gemini configurado.
Modelo de TutorBot: gemini-3.1-pro-preview
Modelo de routing: gemini-3.1-pro-preview
Modelo juez: gemini-3.6-flash


## 4. Modelo de embeddings (BGE-M3)

BGE-M3 se utiliza para representar las consultas y los fragmentos del corpus en un espacio vectorial.
Los vectores se normalizan en L2 y se ejecuta el modelo en GPU cuando CUDA está disponible.


In [8]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from langchain_core.embeddings.embeddings import Embeddings


class SentenceEmbeddings(Embeddings):
    def __init__(self, model_name, batch_size=16):
        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        self.model_name = model_name
        self.batch_size = batch_size

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name
        )

        self.model = AutoModel.from_pretrained(
            model_name
        ).to(self.device).eval()

        print(
            f"{model_name} cargado en {self.device}"
        )

    def _encode(self, textos):
        vectores = []

        for i in range(
            0,
            len(textos),
            self.batch_size
        ):
            lote = textos[
                i:i + self.batch_size
            ]

            entradas = self.tokenizer(
                lote,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                salida = self.model(
                    **entradas
                )

            emb = F.normalize(
                salida.last_hidden_state[:, 0],
                p=2,
                dim=1
            )

            vectores.extend(
                emb.cpu().tolist()
            )

        return vectores

    def embed_documents(self, texts):
        return self._encode(
            list(texts)
        )

    def embed_query(self, text):
        return self._encode(
            [text]
        )[0]


sentence_embeddings = SentenceEmbeddings(
    "BAAI/bge-m3"
)


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BAAI/bge-m3 cargado en cuda


## 5. Índices vectoriales por tipo de contenido

La búsqueda se realiza sobre un índice correspondiente al tipo de contenido que necesita cada
herramienta. De esta forma, una consulta sobre código se compara directamente con chunks de código,
mientras que una consulta conceptual se compara con slides y transcripciones.

Se mantienen cuatro espacios de búsqueda:

- `concepto`: `slide` + `transcripcion`
- `codigo`: `codigo`
- `tarea`: `codigo` + `administrativo`
- `todos`: corpus completo


In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document

documentos = [
    Document(
        page_content=c["texto"],
        metadata={
            **c["metadata"],
            "id": c["id"]
        }
    )
    for c in chunks
]


TIPOS_INDICE = {
    "concepto": {
        "slide",
        "transcripcion"
    },
    "codigo": {
        "codigo"
    },
    "tarea": {
        "slide",
        "transcripcion",
        "codigo",
        "administrativo"
    },
    "todos": None,
}


def documentos_por_tipos(tipos):
    if tipos is None:
        return documentos

    return [
        d for d in documentos
        if d.metadata.get("tipo") in tipos
    ]


indices = {}

for nombre, tipos in TIPOS_INDICE.items():
    docs_indice = documentos_por_tipos(
        tipos
    )

    print(
        f"Construyendo índice '{nombre}': "
        f"{len(docs_indice)} documentos"
    )

    indices[nombre] = FAISS.from_documents(
        docs_indice,
        sentence_embeddings,
        distance_strategy=DistanceStrategy.COSINE
    )

print("\nÍndices listos.")


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Construyendo índice 'concepto': 11989 documentos
Construyendo índice 'codigo': 3402 documentos
Construyendo índice 'tarea': 15412 documentos
Construyendo índice 'todos': 15412 documentos

Índices listos.


In [10]:
# Verificación de distancias en el índice conceptual.
# Una distancia menor representa mayor cercanía.

for doc, score in indices["concepto"].similarity_search_with_score(
    "¿qué es un intent?",
    k=3
):
    print(
        f"distancia={score:.3f}  "
        f"[{doc.metadata['tipo']:14}] "
        f"{doc.page_content[:75]}..."
    )


distancia=0.641  [transcripcion ] Resulta que la intención eh aunque sea redundante, representa la intención ...
distancia=0.707  [transcripcion ] mi respuesta, sí o sí debes generar este campo de aquí. ¿Y cuál es ese camp...
distancia=0.716  [slide         ] **Intención (Intent):** Representa <mark>la intención del usuario</mark>, a...


## 6. Recuperación y control de alcance

La evaluación del retriever utiliza `K_EVALUACION = 3`. Las respuestas de TutorBot utilizan
`K_GENERACION = 5`, de modo que el generador puede recibir más contexto sin modificar las métricas
oficiales del retriever.

El umbral de distancia se utiliza para decidir si una pregunta conceptual dispone de evidencia
suficiente en el corpus.


In [11]:
K_EVALUACION = 3
K_GENERACION = 5

UMBRAL_DISTANCIA = 0.55


MAPA_INDICE = {
    frozenset({
        "slide",
        "transcripcion"
    }): "concepto",

    frozenset({
        "codigo"
    }): "codigo",

    frozenset({
        "slide",
        "transcripcion",
        "codigo",
        "administrativo"
    }): "tarea",
}


def obtener_indice(tipos=None):
    if not tipos:
        return indices["todos"]

    clave = frozenset(tipos)

    if clave not in MAPA_INDICE:
        raise ValueError(
            f"No existe un índice configurado "
            f"para los tipos: {sorted(tipos)}"
        )

    return indices[
        MAPA_INDICE[clave]
    ]


def recuperar(
    consulta,
    tipos=None,
    k=K_GENERACION
):
    """Devuelve [(Document, distancia)] desde el índice correspondiente."""

    db_actual = obtener_indice(
        tipos
    )

    return db_actual.similarity_search_with_score(
        consulta,
        k=k
    )


def formatear_contexto(resultados):
    partes = []

    for doc, distancia in resultados:
        meta = doc.metadata

        origen = (
            f"Ciclo {meta.get('ciclo')} · "
            f"{meta.get('curso', 'Curso no identificado')} · "
            f"{meta.get('fuente', 'Material académico')}"
        )

        if meta.get("clase") is not None:
            origen += f" · clase {meta['clase']}"

        partes.append(
            f"[Fuente: {origen} | distancia {distancia:.3f}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(partes)


### Estimación del umbral de distancia

El cálculo utiliza preguntas conceptuales con referencia en el corpus y consultas identificadas como
fuera de alcance. El valor obtenido se usa como parámetro de control de alcance para
`responder_concepto`.


In [12]:
import numpy as np

positivos_umbral = [
    c for c in eval_set
    if c.get("herramienta_esperada") == "responder_concepto"
    and c.get("chunks_relevantes")
]

negativos_umbral = [
    c for c in eval_set
    if c.get("categoria") == "fuera_de_alcance"
]


def top1_concepto(pregunta):
    resultados = recuperar(
        pregunta,
        tipos={
            "slide",
            "transcripcion"
        },
        k=1
    )

    if not resultados:
        return np.inf

    return float(
        resultados[0][1]
    )


dist_pos = [
    top1_concepto(
        c["pregunta"]
    )
    for c in positivos_umbral
]

dist_neg = [
    top1_concepto(
        c["pregunta"]
    )
    for c in negativos_umbral
]


if dist_pos and dist_neg:
    valores = sorted(
        set(
            dist_pos
            + dist_neg
        )
    )

    candidatos = []

    if valores:
        candidatos.append(
            valores[0] - 1e-6
        )

        for a, b in zip(
            valores[:-1],
            valores[1:]
        ):
            candidatos.append(
                (a + b) / 2
            )

        candidatos.append(
            valores[-1] + 1e-6
        )

    mejor = None

    for umbral in candidatos:
        tpr = np.mean([
            d <= umbral
            for d in dist_pos
        ])

        tnr = np.mean([
            d > umbral
            for d in dist_neg
        ])

        balanced_acc = (
            tpr + tnr
        ) / 2

        registro = (
            balanced_acc,
            umbral,
            tpr,
            tnr
        )

        if (
            mejor is None
            or registro[0] > mejor[0]
        ):
            mejor = registro

    _, UMBRAL_DISTANCIA, tpr_umbral, tnr_umbral = mejor

    print(
        f"UMBRAL_DISTANCIA = "
        f"{UMBRAL_DISTANCIA:.4f}"
    )

    print(
        f"Cobertura de consultas con material: "
        f"{tpr_umbral:.3f}"
    )

    print(
        f"Detección de fuera de alcance: "
        f"{tnr_umbral:.3f}"
    )

else:
    print(
        "No hay suficientes casos para estimar el umbral. "
        f"Se mantiene {UMBRAL_DISTANCIA:.3f}."
    )


UMBRAL_DISTANCIA = 0.9243
Cobertura de consultas con material: 1.000
Detección de fuera de alcance: 1.000


## 7. Herramientas

| Herramienta | Procedimiento | Corpus |
|---|---|---|
| `responder_concepto` | Recuperación conceptual + respuesta fundamentada + control de alcance | slide, transcripcion |
| `explicar_codigo` | Recuperación sobre celdas de notebook + explicación del código | codigo |
| `guiar_tarea` | Solicitud de contexto faltante + guía basada en material | codigo, administrativo |
| `pedir_aclaracion` | Pregunta de desambiguación sin recuperación documental | — |


In [13]:
def _generar(
    system_prompt,
    user_prompt,
    temperature=0.2,
    model=None
):
    modelo = model or MODEL_CHAT

    config_kwargs = {
        "system_instruction":
            system_prompt
    }

    if not modelo.startswith(
        "gemini-3"
    ):
        config_kwargs[
            "temperature"
        ] = temperature

    config = GenerateContentConfig(
        **config_kwargs
    )

    mensajes = [
        genai.types.Content(
            role="user",
            parts=[
                genai.types.Part.from_text(
                    text=user_prompt
                )
            ]
        )
    ]

    return gemini_client.models.generate_content(
        model=modelo,
        contents=mensajes,
        config=config
    ).text


ETIQUETA_FALLBACK = "[Fallback]"


def fn_fallback(
    consulta: str,
    motivo: str = "fuera_alcance"
) -> str:
    """Respuesta de respaldo cuando no existe evidencia suficiente."""

    if motivo == "fuera_alcance":
        return (
            f"{ETIQUETA_FALLBACK} "
            "No encontré evidencia suficiente en los materiales académicos disponibles "
            "para responder esa consulta. Puedo ayudarte con conceptos, "
            "código, notebooks y entregables del curso de Diseño de "
            "Chatbots Conversacionales."
        )

    if motivo == "sin_codigo":
        return (
            f"{ETIQUETA_FALLBACK} "
            "No encontré fragmentos de código relacionados en los "
            "notebooks disponibles. Indícame el notebook, función o "
            "fragmento que quieres revisar."
        )

    if motivo == "sin_guia":
        return (
            f"{ETIQUETA_FALLBACK} "
            "No encontré material suficiente para construir una guía "
            "sobre esa tarea. Indícame la clase, notebook o entregable "
            "específico."
        )

    if motivo == "mensaje_vacio":
        return (
            f"{ETIQUETA_FALLBACK} "
            "Escribe una consulta para poder ayudarte."
        )

    return (
        f"{ETIQUETA_FALLBACK} "
        "No pude completar la consulta en este momento. "
        "Intenta nuevamente o reformula la pregunta."
    )


def fn_responder_concepto(
    pregunta: str
) -> str:

    resultados = recuperar(
        pregunta,
        tipos={
            "slide",
            "transcripcion"
        },
        k=K_GENERACION
    )

    suficiente = (
        bool(resultados)
        and resultados[0][1]
        <= UMBRAL_DISTANCIA
    )

    if suficiente:
        system = (
            "Eres TutorBot, asistente del curso de "
            "Diseño de Chatbots Conversacionales. "
            "Responde en español de forma clara y directa. "
            "Usa únicamente la documentación proporcionada. "
            "No agregues información que no esté sustentada por ella. "
            "Indica la fuente o clase cuando sea útil."
        )

        user = (
            f"Pregunta: {pregunta}\n\n"
            f"Documentación:\n"
            f"{formatear_contexto(resultados)}"
        )

        return _generar(
            system,
            user
        )

    return fn_fallback(
        pregunta,
        motivo="fuera_alcance"
    )


def fn_explicar_codigo(
    pregunta: str
) -> str:

    resultados = recuperar(
        pregunta,
        tipos={"codigo"},
        k=K_GENERACION
    )

    if not resultados:
        return fn_fallback(
            pregunta,
            motivo="sin_codigo"
        )

    system = (
        "Eres TutorBot. Explica el código del material académico en español. "
        "Usa únicamente los fragmentos proporcionados. "
        "Describe qué hace el código y cómo intervienen sus "
        "componentes. Indica el notebook de origen. "
        "No inventes fragmentos que no estén en la documentación."
    )

    user = (
        f"Consulta: {pregunta}\n\n"
        f"Fragmentos de código:\n"
        f"{formatear_contexto(resultados)}"
    )

    return _generar(
        system,
        user
    )


def fn_guiar_tarea(
    tarea: str,
    detalle: str = ""
) -> str:

    if not detalle.strip():
        return (
            f"Para guiarte con '{tarea}' necesito un dato más: "
            "¿sobre qué clase, notebook o entregable "
            "específico lo necesitas?"
        )

    consulta = (
        f"{tarea} {detalle}"
    )

    resultados = recuperar(
        consulta,
        tipos={
            "slide",
            "transcripcion",
            "codigo",
            "administrativo"
        },
        k=K_GENERACION
    )

    if not resultados:
        return fn_fallback(
            consulta,
            motivo="sin_guia"
        )

    system = (
        "Eres TutorBot. Entrega una guía numerada paso a paso "
        "en español, basada únicamente en el material provisto. "
        "Usa un máximo de 6 pasos concretos y accionables."
    )

    user = (
        f"Tarea: {tarea}\n"
        f"Contexto del estudiante: {detalle}\n\n"
        f"Material:\n"
        f"{formatear_contexto(resultados)}"
    )

    return _generar(
        system,
        user
    )


def fn_pedir_aclaracion(
    consulta: str
) -> str:

    system = (
        "Eres TutorBot. La consulta del estudiante es ambigua. "
        "Devuelve una sola pregunta corta en español "
        "para desambiguarla. No respondas la consulta."
    )

    return _generar(
        system,
        f"Consulta ambigua: {consulta}",
        temperature=0.3
    )


print(
    "Funciones definidas: concepto, código, tarea, "
    "aclaración y fallback."
)


Funciones definidas: concepto, código, tarea, aclaración y fallback.


## 8. Servidor MCP

Cada función se expone como herramienta MCP. Las descripciones de las herramientas indican al agente
en qué situaciones debe utilizar cada una.


In [14]:
from fastmcp import FastMCP, Client

mcp_server = FastMCP(
    "tutorbot"
)


@mcp_server.tool()
def responder_concepto(
    pregunta: str
) -> str:
    """Responde preguntas conceptuales o teóricas, incluidas consultas claras
    que puedan quedar fuera del contenido del curso.

    Úsala para definiciones, conceptos y preguntas de conocimiento:
    intent, utterance, slot, entidad, NLU, NLG, Dialogue Manager,
    stemming, stopwords, n-gramas, Transformers, beam search y RAG.

    Si la pregunta es clara pero el corpus no contiene evidencia suficiente,
    esta herramienta aplica el control de alcance.

    No la uses para explicar una función o fragmento de código concreto.
    """
    return fn_responder_concepto(
        pregunta
    )


@mcp_server.tool()
def explicar_codigo(
    pregunta: str
) -> str:
    """Explica funciones, clases, librerías, celdas y fragmentos existentes
    en los notebooks del curso.

    Úsala también cuando la pregunta empiece por 'cómo' si el estudiante
    está preguntando cómo funciona, se define, se configura o se usa
    un elemento de código ya presente en los notebooks.

    Ejemplos: CountVectorizer, predict_intent, FastAPI, torch, RASA.
    """
    return fn_explicar_codigo(
        pregunta
    )


@mcp_server.tool()
def guiar_tarea(
    tarea: str,
    detalle: str = ""
) -> str:
    """Guía al estudiante cuando expresa intención de crear, modificar,
    implementar, desplegar o completar una tarea.

    También se utiliza para preguntas sobre qué exige un entregable
    o una rúbrica.

    No la selecciones solo porque una pregunta empiece por 'cómo'.
    Si pregunta cómo funciona código existente, corresponde explicar_codigo.
    """
    return fn_guiar_tarea(
        tarea,
        detalle
    )


@mcp_server.tool()
def pedir_aclaracion(
    consulta: str
) -> str:
    """Pide aclaración únicamente cuando el mensaje es ambiguo y no permite
    determinar la intención del estudiante.

    Una pregunta clara fuera del alcance del curso no es ambigua y debe
    dirigirse a responder_concepto para que se aplique el control de alcance.
    """
    return fn_pedir_aclaracion(
        consulta
    )


mcp_client = Client(
    mcp_server
)

print(
    "Herramientas registradas: "
    "responder_concepto, explicar_codigo, "
    "guiar_tarea, pedir_aclaracion"
)


Herramientas registradas: responder_concepto, explicar_codigo, guiar_tarea, pedir_aclaracion


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Agente

Gemini selecciona una herramienta y devuelve al estudiante la respuesta generada por esa herramienta.
El agente conserva un mecanismo de reintento ante errores de ejecución.


In [15]:
import asyncio


SYSTEM_AGENTE = """
Eres TutorBot, asistente académico del diplomado de Desarrollo
de Aplicaciones con IA.

Selecciona siempre una herramienta.

MEMORIA DE CONVERSACIÓN:

- Antes de seleccionar una herramienta, revisa el historial reciente.
- Resuelve referencias como "eso", "ese", "esa", "cada uno",
  "el anterior", "el modelo", "la función", "cómo lo hago",
  "dame ejemplos" o expresiones similares usando el historial.
- Si la consulta puede entenderse usando el historial,
  NO uses pedir_aclaracion.
- Al invocar una herramienta, convierte la consulta en una
  pregunta autosuficiente cuando sea necesario.

Ejemplo:

Historial:
Usuario: ¿Qué diferencia hay entre aprendizaje supervisado
y no supervisado?
TutorBot: ...

Nueva consulta:
Dame ejemplos de cada uno

La herramienta debe recibir algo equivalente a:
"Dame ejemplos de aprendizaje supervisado y aprendizaje no supervisado."

Reglas de selección:

1. responder_concepto
   - teoría, definiciones y conceptos;
   - preguntas claras de conocimiento;
   - preguntas de seguimiento sobre conceptos mencionados
     anteriormente en la conversación;
   - el control de alcance se realiza dentro de la herramienta.

2. explicar_codigo
   - funciones, clases, librerías, celdas o fragmentos
     existentes en los notebooks;
   - preguntas de seguimiento sobre código mencionado
     anteriormente;
   - una pregunta que empieza por "cómo" sigue siendo de código
     si pregunta cómo funciona, se define, se configura o se usa
     código existente.

3. guiar_tarea
   - el estudiante expresa intención de crear, modificar,
     implementar, desplegar o completar algo;
   - preguntas sobre requisitos de una actividad o entregable;
   - preguntas de seguimiento sobre una tarea previamente
     mencionada.

4. pedir_aclaracion
   - úsala solamente si la consulta sigue siendo ambigua
     después de revisar el historial;
   - no la uses si el referente puede deducirse de mensajes
     anteriores;
   - no la uses para una pregunta clara solo porque esté
     fuera del alcance del corpus.

El fallback se activa automáticamente cuando no existe evidencia
suficiente en el corpus o cuando la interacción no puede completarse.

Devuelve la respuesta de la herramienta en español.
Si contiene la etiqueta [Fallback], consérvala.
"""


def extraer_texto_gradio(
    content
):
    """
    Convierte el contenido de mensajes de Gradio
    a texto simple.
    """

    if isinstance(
        content,
        str
    ):
        return content.strip()

    if isinstance(
        content,
        list
    ):
        partes = []

        for bloque in content:

            if isinstance(
                bloque,
                str
            ):
                partes.append(
                    bloque
                )

            elif isinstance(
                bloque,
                dict
            ):
                if (
                    bloque.get("type")
                    == "text"
                ):
                    texto = bloque.get(
                        "text",
                        ""
                    )

                    if texto:
                        partes.append(
                            str(texto)
                        )

        return "\n".join(
            partes
        ).strip()

    if isinstance(
        content,
        dict
    ):
        if (
            content.get("type")
            == "text"
        ):
            return str(
                content.get(
                    "text",
                    ""
                )
            ).strip()

    return ""


def normalizar_historial_gradio(
    historial,
    max_mensajes=8
):
    """
    Convierte el historial de Gradio 6
    al formato simple que utiliza TutorBot.
    """

    historial = (
        historial
        or []
    )[-max_mensajes:]

    normalizado = []

    for mensaje in historial:

        if not isinstance(
            mensaje,
            dict
        ):
            continue

        role = mensaje.get(
            "role",
            ""
        )

        if role not in {
            "user",
            "assistant",
            "model"
        }:
            continue

        content = extraer_texto_gradio(
            mensaje.get(
                "content",
                ""
            )
        )

        if not content:
            continue

        normalizado.append({
            "role":
                role,

            "content":
                content,
        })

    return normalizado


def construir_mensajes(
    consulta,
    historial=None,
    max_mensajes=8
):
    mensajes = []

    historial = (
        historial
        or []
    )[-max_mensajes:]

    for turno in historial:
        role = turno.get(
            "role",
            "user"
        )

        contenido = turno.get(
            "content",
            ""
        )

        if role == "assistant":
            role = "model"

        if role not in {
            "user",
            "model"
        }:
            continue

        if not isinstance(
            contenido,
            str
        ):
            continue

        if not contenido.strip():
            continue

        mensajes.append(
            genai.types.Content(
                role=role,
                parts=[
                    genai.types.Part.from_text(
                        text=contenido
                    )
                ]
            )
        )

    mensajes.append(
        genai.types.Content(
            role="user",
            parts=[
                genai.types.Part.from_text(
                    text=consulta
                )
            ]
        )
    )

    return mensajes


async def agente(
    consulta,
    historial=None,
    max_iterations=3
):
    consulta = (
        consulta
        or ""
    ).strip()

    if not consulta:
        return (
            fn_fallback(
                "",
                "mensaje_vacio"
            ),
            ["fallback"]
        )

    for intento in range(
        1,
        max_iterations + 1
    ):
        try:
            async with mcp_client:
                config = GenerateContentConfig(
                    system_instruction=(
                        SYSTEM_AGENTE
                    ),
                    tools=[
                        mcp_client.session
                    ],
                )

                respuesta = (
                    await gemini_client.aio.models
                    .generate_content(
                        model=MODEL_ROUTING,
                        contents=construir_mensajes(
                            consulta,
                            historial
                        ),
                        config=config
                    )
                )

            usadas = []

            for contenido in (
                respuesta.automatic_function_calling_history
                or []
            ):
                for parte in (
                    contenido.parts
                    or []
                ):
                    if getattr(
                        parte,
                        "function_call",
                        None
                    ):
                        usadas.append(
                            parte.function_call.name
                        )

            texto = (
                respuesta.text
                or ""
            ).strip()

            if not texto:
                raise ValueError(
                    "respuesta vacía"
                )

            return (
                texto,
                usadas
            )

        except Exception as e:
            if intento == max_iterations:
                return (
                    fn_fallback(
                        consulta,
                        "error_temporal"
                    ),
                    ["fallback"]
                )

            mensaje = str(
                e
            )

            if (
                "429" in mensaje
                or "RESOURCE_EXHAUSTED" in mensaje
                or "503" in mensaje
                or "UNAVAILABLE" in mensaje
            ):
                espera = min(
                    10 * intento,
                    60
                )
            else:
                espera = (
                    1.5 * intento
                )

            await asyncio.sleep(
                espera
            )


def historial_a_texto(
    historial,
    max_mensajes=6
):
    historial = (
        historial
        or []
    )[-max_mensajes:]

    partes = []

    for mensaje in historial:

        role = mensaje.get(
            "role",
            ""
        )

        content = mensaje.get(
            "content",
            ""
        )

        if not content:
            continue

        if role == "user":
            etiqueta = "Usuario"

        elif role in {
            "assistant",
            "model"
        }:
            etiqueta = "TutorBot"

        else:
            continue

        partes.append(
            f"{etiqueta}: {content}"
        )

    return "\n".join(
        partes
    )


async def contextualizar_consulta(
    consulta,
    historial
):
    """
    Convierte una pregunta de seguimiento en una consulta
    autosuficiente antes de enviarla al router y al RAG.
    """

    if not historial:
        return consulta

    historial_texto = historial_a_texto(
        historial
    )

    if not historial_texto:
        return consulta

    system = """
Eres un componente de contextualización de consultas
para un asistente académico.

Tu única tarea es reescribir la CONSULTA ACTUAL como una
consulta autosuficiente utilizando el HISTORIAL.

Reglas:

- NO respondas la pregunta.
- Devuelve únicamente la consulta reescrita.
- Resuelve referencias como:
  "eso", "ese", "esa", "ellos", "cada uno",
  "el anterior", "la anterior", "ese modelo",
  "esa función", "dame ejemplos", "¿y cómo?",
  "¿y por qué?", "¿y cuál es su fórmula?"
  usando el historial.
- Conserva la intención original del usuario.
- No inventes información que no aparezca en el historial.
- Si la consulta ya es autosuficiente, devuélvela sin cambios.
"""

    user = (
        f"HISTORIAL:\n"
        f"{historial_texto}\n\n"
        f"CONSULTA ACTUAL:\n"
        f"{consulta}"
    )

    try:
        respuesta = (
            await gemini_client.aio.models.generate_content(
                model=MODEL_ROUTING,
                contents=[
                    genai.types.Content(
                        role="user",
                        parts=[
                            genai.types.Part.from_text(
                                text=user
                            )
                        ]
                    )
                ],
                config=GenerateContentConfig(
                    system_instruction=system
                )
            )
        )

        contextualizada = (
            respuesta.text
            or ""
        ).strip()

        return (
            contextualizada
            if contextualizada
            else consulta
        )

    except Exception as e:
        print(
            "No se pudo contextualizar:",
            e
        )

        return consulta


## 10. Demostración de los flujos

Las consultas siguientes usan materiales de cursos diferentes y muestran los flujos de
concepto, código, guía, aclaración y fallback.


In [16]:
respuesta, usadas = await agente(
    "¿Qué es una CNN y por qué se usa en visión artificial?"
)
print("Herramientas:", usadas)
print(respuesta)


/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py:744: DeprecationWarning: Inheritance class AiohttpClientSession from ClientSession is discouraged
  class AiohttpClientSession(aiohttp.ClientSession):  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Herramientas: ['responder_concepto']
¡Hola! Soy TutorBot. Basado en la documentación proporcionada, te explico qué es una CNN y por qué se utiliza en visión artificial:

**¿Qué es una CNN?**
Una CNN (Red Neuronal Convolucional) es un modelo altamente eficiente diseñado para aprender descripciones a partir de patrones. Su arquitectura tiene dos etapas principales: el aprendizaje de características (*Feature Learning*) y una red completamente conectada (*Fully-connected Net*). Ingresa la imagen cruda y aprende sus características de forma jerárquica (*Fuente: Ciclo 2 · Desarrollo de Aplicaciones con Visión Artificial · clase 5*).

**¿Por qué se usa en visión artificial (imágenes)?**
La CNN es considerada una red "especialista en imágenes" (*Fuente: Ciclo 3 · Redes Neuronales para el Análisis de Series Temporales · clase 3*) por las siguientes razones:

1. **Conserva la posición espacial:** A diferencia de otros modelos (como el LSTM tradicional) que aplanan los números y pierden la infor

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
respuesta, usadas = await agente(
    "¿Qué hace la clase QLearningAgent?"
)
print("Herramientas:", usadas)
print(respuesta)


Herramientas: ['explicar_codigo']
¡Hola! Soy TutorBot. Con base en los fragmentos proporcionados, te explico el funcionamiento de esta clase.

La clase **`QLearningAgent`** implementa un agente de aprendizaje por refuerzo basado en el algoritmo Q-learning. Su objetivo principal es aprender a tomar las mejores decisiones (acciones) dentro de un entorno para maximizar sus recompensas, equilibrando la exploración de nuevos estados y la explotación de lo que ya ha aprendido.

El código de esta clase proviene del notebook sobre Q-learning (Inteligencia Artificial para Juegos).

A continuación, te explico qué hace y cómo intervienen sus distintos componentes:

### 1. Inicialización (`__init__`)
Cuando se instancia el agente, recibe los parámetros del entorno (definidos en un `mdp` o Proceso de Decisión de Markov) y variables de configuración. Aquí inicializa:
*   **Parámetros del entorno:** El factor de descuento (`gamma`), los estados terminales (`terminals`) y las acciones posibles (`all_a

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [18]:
respuesta, usadas = await agente(
    "Quiero crear un dashboard con Streamlit"
)
print("Herramientas:", usadas)
print(respuesta)


Herramientas: ['guiar_tarea']
Para guiarte con 'crear un dashboard con Streamlit' necesito un dato más: ¿sobre qué clase, notebook o entregable específico lo necesitas?


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
respuesta, usadas = await agente(
    "Explícame el modelo"
)
print("Herramientas:", usadas)
print(respuesta)


Herramientas: ['pedir_aclaracion', 'responder_concepto']
Hola. Basado en los documentos del curso, te explico los conceptos principales sobre los modelos Transformers:

*   **Arquitectura general:** Los Transformers son una arquitectura base de la que derivan modelos de inteligencia artificial muy conocidos, como BERT, GPT o T5. Todos parten de la misma idea, pero se diferencian en particularidades como la cantidad de capas, *encoders* (codificadores) y *decoders* (decodificadores) que utilizan *(Ciclo 2, Clase 7)*. Estos modelos representan una evolución frente a las redes clásicas (*Fully Connected*) que no podían simular secuencias de tiempo de forma natural, y las redes neuronales recurrentes (RNN).
*   **Mecanismo de Atención:** La característica principal de los Transformers es que utilizan distintos tipos de "mecanismos de atención". Este mecanismo aprende los pesos y relaciones de las palabras de salida respecto a las de entrada. Por ejemplo, en una tarea de traducción, el mode

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
respuesta, usadas = await agente(
    "¿Quién ganó la Copa del Mundo de 2022?"
)
print("Herramientas:", usadas)
print(respuesta)
assert ETIQUETA_FALLBACK in respuesta


Herramientas: ['responder_concepto']
[Fallback] No encontré evidencia suficiente en los materiales académicos disponibles para responder esa consulta. Puedo ayudarte con conceptos, código, notebooks y entregables del curso de Diseño de Chatbots Conversacionales.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 11. Interfaz con Gradio

La interfaz permite consultar los materiales de todos los cursos incluidos en el corpus.


In [21]:
import gradio as gr


async def responder_gradio(
    message,
    history
):
    # 1. Convierte el follow-up en una consulta autosuficiente
    consulta_contextual = (
        await contextualizar_consulta(
            message,
            history
        )
    )

    print(
        "Consulta original:",
        message
    )

    print(
        "Consulta contextualizada:",
        consulta_contextual
    )

    # 2. El agente recibe la consulta ya resuelta
    respuesta, _ = await agente(
        consulta_contextual,
        historial=history
    )

    return respuesta


demo = gr.ChatInterface(
    fn=responder_gradio,
    title="TutorBot",
    description=(
        "Asistente académico del curso de "
        "Diseño de Chatbots Conversacionales"
    ),
    examples=[
        "¿Qué diferencia hay entre aprendizaje supervisado y no supervisado?",
        "¿Qué hace la clase QLearningAgent?",
        "Quiero crear un dashboard con Streamlit",
        "¿Qué es una CNN?",
        "Quiero implementar RAG en mi chatbot",
        "¿Qué caracteriza a una serie temporal?",
    ],
    save_history=True,
)


print("Interfaz Gradio preparada.")


# Para probarla desde Colab:
# demo.launch(share=True, debug=False)


Interfaz Gradio preparada.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 12. Evaluación del retriever (R)

La relevancia se evalúa a nivel de chunk. Un documento recuperado cuenta como relevante si su
`chunk_id` aparece entre las referencias del caso de evaluación.

Las métricas principales son Hit Rate@3, MRR, Mean Precision@3 y Mean Recall@3.


In [22]:
import numpy as np

TIPOS_POR_HERRAMIENTA = {
    "responder_concepto": {
        "slide",
        "transcripcion"
    },
    "explicar_codigo": {
        "codigo"
    },
    "guiar_tarea": {
        "slide",
        "transcripcion",
        "codigo",
        "administrativo"
    },
}

casos_r = [
    c for c in eval_set
    if c.get("chunks_relevantes")
]

recuperados_por_caso = []

for caso in casos_r:
    tipos = TIPOS_POR_HERRAMIENTA[
        caso["herramienta_esperada"]
    ]

    resultados = recuperar(
        caso["pregunta"],
        tipos=tipos,
        k=K_EVALUACION
    )

    recuperados_por_caso.append(
        [
            d.metadata["chunk_id"]
            for d, _ in resultados
        ]
    )

print(
    f"Casos evaluados: "
    f"{len(casos_r)} | "
    f"k = {K_EVALUACION}"
)


Casos evaluados: 27 | k = 3


In [23]:
hit, rr, prec, rec = [], [], [], []

for caso, recuperados in zip(
    casos_r,
    recuperados_por_caso
):
    relevantes = {
        c["chunk_id"]
        for c in caso[
            "chunks_relevantes"
        ]
    }

    aciertos = [
        1 if cid in relevantes else 0
        for cid in recuperados
    ]

    hit.append(
        1 if any(aciertos) else 0
    )

    rr.append(
        next(
            (
                1 / (i + 1)
                for i, a in enumerate(
                    aciertos
                )
                if a
            ),
            0
        )
    )

    prec.append(
        sum(aciertos)
        / len(recuperados)
        if recuperados
        else 0
    )

    rec.append(
        sum(aciertos)
        / len(relevantes)
    )

print(
    f"Hit Rate@{K_EVALUACION}:       "
    f"{np.mean(hit):.3f}"
)

print(
    f"MRR:              "
    f"{np.mean(rr):.3f}"
)

print(
    f"Mean Precision@{K_EVALUACION}: "
    f"{np.mean(prec):.3f}"
)

print(
    f"Mean Recall@{K_EVALUACION}:    "
    f"{np.mean(rec):.3f}"
)


Hit Rate@3:       0.444
MRR:              0.315
Mean Precision@3: 0.185
Mean Recall@3:    0.185


In [24]:
from collections import defaultdict

# Desglose por categoría
por_cat = defaultdict(
    list
)

for caso, h in zip(
    casos_r,
    hit
):
    por_cat[
        caso["categoria"]
    ].append(h)

print("Hit Rate por categoría:")

for cat, valores in por_cat.items():
    print(
        f"{cat:16} "
        f"Hit Rate@{K_EVALUACION} = "
        f"{np.mean(valores):.3f} "
        f"({len(valores)} casos)"
    )


# Diagnóstico para distintos valores de k
print("\nHit Rate según k:")

for k_diag in [1, 3, 5]:
    hits_k = []

    for caso in casos_r:
        tipos = TIPOS_POR_HERRAMIENTA[
            caso["herramienta_esperada"]
        ]

        resultados = recuperar(
            caso["pregunta"],
            tipos=tipos,
            k=k_diag
        )

        recuperados = {
            d.metadata["chunk_id"]
            for d, _ in resultados
        }

        relevantes = {
            ref["chunk_id"]
            for ref in caso[
                "chunks_relevantes"
            ]
        }

        hits_k.append(
            int(
                bool(
                    recuperados
                    & relevantes
                )
            )
        )

    print(
        f"  Hit Rate@{k_diag}: "
        f"{np.mean(hits_k):.3f}"
    )


Hit Rate por categoría:
conceptual       Hit Rate@3 = 0.222 (9 casos)
codigo           Hit Rate@3 = 0.889 (9 casos)
tarea            Hit Rate@3 = 0.222 (9 casos)

Hit Rate según k:
  Hit Rate@1: 0.222
  Hit Rate@3: 0.444
  Hit Rate@5: 0.519


## 13. Evaluación del enrutamiento

La métrica compara la herramienta seleccionada por Gemini con la herramienta esperada en cada caso.
Durante esta evaluación se solicita únicamente la selección de la función; las herramientas no se
ejecutan.


In [25]:
import asyncio
import numpy as np
from google.genai import types


declaraciones = [
    types.FunctionDeclaration(
        name="responder_concepto",
        description=(
            "Selecciona esta herramienta para preguntas conceptuales, "
            "teóricas o de conocimiento. También para preguntas claras "
            "que puedan estar fuera del contenido del curso; la propia "
            "herramienta aplicará el control de alcance."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "pregunta": {
                    "type": "string"
                }
            },
            "required": [
                "pregunta"
            ]
        }
    ),

    types.FunctionDeclaration(
        name="explicar_codigo",
        description=(
            "Selecciona esta herramienta para preguntas sobre funciones, "
            "clases, librerías, celdas o código existente en los notebooks. "
            "Una pregunta que empiece por 'cómo' pertenece aquí si pregunta "
            "cómo funciona, se define, se configura o se usa código existente."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "pregunta": {
                    "type": "string"
                }
            },
            "required": [
                "pregunta"
            ]
        }
    ),

    types.FunctionDeclaration(
        name="guiar_tarea",
        description=(
            "Selecciona esta herramienta cuando el estudiante expresa "
            "intención de crear, modificar, implementar, desplegar o "
            "completar una tarea, o cuando pregunta qué exige un entregable "
            "o una rúbrica. No la selecciones solo por la palabra 'cómo'."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "tarea": {
                    "type": "string"
                },
                "detalle": {
                    "type": "string"
                }
            },
            "required": [
                "tarea"
            ]
        }
    ),

    types.FunctionDeclaration(
        name="pedir_aclaracion",
        description=(
            "Selecciona esta herramienta solo si el mensaje es ambiguo, "
            "incompleto o demasiado corto para identificar la intención. "
            "No la selecciones para preguntas claras fuera del alcance."
        ),
        parameters_json_schema={
            "type": "object",
            "properties": {
                "consulta": {
                    "type": "string"
                }
            },
            "required": [
                "consulta"
            ]
        }
    ),
]


tools_routing = types.Tool(
    function_declarations=
        declaraciones
)


config_routing = (
    types.GenerateContentConfig(
        system_instruction="""
Eres el router de TutorBot.
Selecciona exactamente una herramienta.

Criterios:

- responder_concepto:
  teoría, conceptos, definiciones y preguntas claras de conocimiento.
  Las consultas claras fuera del material también van aquí; el control
  de alcance ocurre dentro de responder_concepto.

- explicar_codigo:
  funciones, clases, librerías, celdas y código ya existente.
  La palabra "cómo" no implica guiar_tarea. Si la pregunta es
  "cómo funciona", "cómo se define", "cómo se configura" o
  "cómo se usa" código existente, selecciona explicar_codigo.

- guiar_tarea:
  el estudiante quiere crear, modificar, implementar, desplegar o
  completar algo, o pregunta por requisitos de un entregable/rúbrica.

- pedir_aclaracion:
  la consulta es realmente ambigua, incompleta o demasiado corta.

No respondas la consulta.
Solo selecciona una herramienta.
""",
        tools=[
            tools_routing
        ],
        tool_config=
            types.ToolConfig(
                function_calling_config=
                    types.FunctionCallingConfig(
                        mode="ANY"
                    )
            )
    )
)


async def seleccionar_herramienta(
    pregunta
):
    respuesta = (
        await gemini_client.aio.models.generate_content(
            model=MODEL_ROUTING,
            contents=pregunta,
            config=config_routing
        )
    )

    llamadas = (
        respuesta.function_calls
    )

    if not llamadas:
        return "ninguna"

    return llamadas[0].name


aciertos_ruta = []
detalle_ruta = []

ESPERA_ROUTING = 4
MAX_REINTENTOS = 5


for i, caso in enumerate(
    eval_set,
    start=1
):
    print(
        f"[{i}/{len(eval_set)}] "
        f"{caso['id']} - "
        f"{caso['pregunta'][:65]}"
    )

    elegida = None

    for intento in range(
        1,
        MAX_REINTENTOS + 1
    ):
        try:
            elegida = (
                await seleccionar_herramienta(
                    caso["pregunta"]
                )
            )
            break

        except Exception as e:
            mensaje = str(e)

            if (
                "429" in mensaje
                or
                "RESOURCE_EXHAUSTED"
                in mensaje
                or
                "503" in mensaje
                or
                "UNAVAILABLE" in mensaje
            ):
                espera = (
                    15 * intento
                )

                print(
                    f"  Cuota temporal alcanzada. "
                    f"Esperando {espera}s..."
                )

                await asyncio.sleep(
                    espera
                )
                continue

            raise

    if elegida is None:
        elegida = "ninguna"

    esperada = (
        caso[
            "herramienta_esperada"
        ]
    )

    ok = int(
        elegida == esperada
    )

    aciertos_ruta.append(
        ok
    )

    detalle_ruta.append(
        (
            caso["id"],
            esperada,
            elegida,
            ok
        )
    )

    print(
        f"  Esperada: {esperada}"
    )
    print(
        f"  Elegida:  {elegida}"
    )
    print(
        f"  Resultado: "
        f"{'CORRECTO' if ok else 'INCORRECTO'}"
    )

    if i < len(eval_set):
        await asyncio.sleep(
            ESPERA_ROUTING
        )


accuracy_ruta = float(
    np.mean(
        aciertos_ruta
    )
)

print(
    "\nAccuracy de selección "
    f"de herramienta: "
    f"{accuracy_ruta:.3f}"
)

print("\nCasos mal enrutados:")

errores_routing = 0

for (
    cid,
    esperada,
    elegida,
    ok
) in detalle_ruta:

    if not ok:
        errores_routing += 1

        print(
            f"  {cid}: "
            f"esperaba {esperada}, "
            f"eligió {elegida}"
        )

if errores_routing == 0:
    print("  Ninguno")


[1/32] EV01 - ¿Cuál es la diferencia entre aprendizaje supervisado y no supervi
  Esperada: responder_concepto
  Elegida:  responder_concepto
  Resultado: CORRECTO
[2/32] EV02 - ¿Para qué se utiliza Pandas al trabajar con datos tabulares?
  Cuota temporal alcanzada. Esperando 15s...
  Cuota temporal alcanzada. Esperando 30s...
  Esperada: responder_concepto
  Elegida:  responder_concepto
  Resultado: CORRECTO
[3/32] EV03 - ¿Cuál es el propósito de una visualización de datos?
  Esperada: responder_concepto
  Elegida:  responder_concepto
  Resultado: CORRECTO
[4/32] EV04 - ¿Por qué se realiza preprocesamiento de textos antes de analizarl
  Cuota temporal alcanzada. Esperando 15s...
  Esperada: responder_concepto
  Elegida:  responder_concepto
  Resultado: CORRECTO
[5/32] EV05 - ¿Qué es una CNN y por qué se usa en visión artificial?
  Esperada: responder_concepto
  Elegida:  responder_concepto
  Resultado: CORRECTO
[6/32] EV06 - ¿Qué relación existe entre un agente y su entorno?
  Esperad

## 14. Evaluación de la generación (AG)

Se evalúa una muestra reproducible mediante:

- Faithfulness
- Answer Relevancy
- Answer Correctness

Las respuestas se generan con `MODEL_CHAT`. Las evaluaciones con LLM utilizan
`MODEL_JUEZ`.

El detalle por caso se guarda en la carpeta local `resultados` e incluye pregunta, respuesta generada, referencia,
contexto recuperado, chunks, distancias y las tres métricas.


In [26]:
def juez(
    system_prompt,
    user_prompt
):
    return _generar(
        system_prompt,
        user_prompt,
        temperature=0.0,
        model=MODEL_JUEZ
    )


SYS_FAITHFULNESS = """
Eres evaluador de un sistema RAG.
Recibes una pregunta, una respuesta generada y la documentación usada.
Identifica las afirmaciones factuales y determina si cada una está
respaldada por la documentación.

Devuelve una línea por afirmación con:
<afirmación> | Respaldada|No respaldada

Al final:
TOTAL <respaldadas>/<total>

No respondas de memoria.
"""


SYS_RELEVANCY = """
Eres evaluador de un sistema RAG.

Verifica:
1. La respuesta contesta directamente la pregunta.
2. La información incluida está relacionada con la pregunta.
3. La respuesta no evade la pregunta ni es vaga.

Responde Sí o No por criterio y al final:
TOTAL <si>/3
"""


SYS_CORRECTNESS = """
Eres evaluador de un sistema RAG.

Compara la respuesta generada con la respuesta correcta de referencia.

Verifica:
1. Contesta correctamente.
2. No contradice la referencia.
3. No omite información importante de la referencia.

Responde Sí o No por criterio y al final:
TOTAL <si>/3
"""


def parsear_total(texto):
    for linea in reversed(
        texto.strip().split("\n")
    ):
        if (
            "TOTAL" in linea.upper()
            and "/" in linea
        ):
            try:
                frac = (
                    linea.upper()
                    .split("TOTAL")[1]
                    .strip()
                    .split()[0]
                )

                num, den = (
                    frac.split("/")
                )

                return (
                    float(num)
                    / float(den)
                )

            except Exception:
                continue

    return None


In [27]:
import random
import time
import numpy as np
import pandas as pd

random.seed(0)

MIN_INTERVAL_SECONDS = 4.0
MAX_REINTENTOS_TEMPORALES = 6
_ultima_llamada_gemini = 0.0


def esperar_turno_gemini():
    global _ultima_llamada_gemini

    ahora = time.monotonic()

    restante = (
        MIN_INTERVAL_SECONDS
        - (
            ahora
            - _ultima_llamada_gemini
        )
    )

    if restante > 0:
        time.sleep(
            restante
        )


def generar_controlado(
    system_prompt,
    user_prompt,
    temperature=0.2,
    model=None
):
    global _ultima_llamada_gemini

    for intento in range(
        1,
        MAX_REINTENTOS_TEMPORALES + 1
    ):
        esperar_turno_gemini()

        try:
            respuesta = _generar(
                system_prompt,
                user_prompt,
                temperature=temperature,
                model=model
            )

            _ultima_llamada_gemini = (
                time.monotonic()
            )

            return respuesta

        except Exception as e:
            _ultima_llamada_gemini = (
                time.monotonic()
            )

            mensaje = str(e)

            # ------------------------------------------
            # Error 429: cuota / rate limit
            # ------------------------------------------
            es_429 = (
                "429" in mensaje
                or
                "RESOURCE_EXHAUSTED" in mensaje
            )

            # ------------------------------------------
            # Error 503: saturación temporal del modelo
            # ------------------------------------------
            es_503 = (
                "503" in mensaje
                or
                "UNAVAILABLE" in mensaje
                or
                "high demand" in mensaje.lower()
            )

            if es_429:
                espera = min(
                    15 * intento,
                    90
                )

                print(
                    f"  Cuota temporal alcanzada "
                    f"(intento {intento}/"
                    f"{MAX_REINTENTOS_TEMPORALES}). "
                    f"Esperando {espera}s..."
                )

                time.sleep(
                    espera
                )

                continue

            if es_503:
                # Backoff exponencial:
                # 10s → 20s → 40s → 80s → 90s...
                espera = min(
                    10 * (
                        2 ** (
                            intento - 1
                        )
                    ),
                    90
                )

                # Pequeño jitter para evitar reintentar
                # exactamente junto con otras solicitudes.
                espera += random.uniform(
                    0,
                    3
                )

                print(
                    f"  Gemini temporalmente saturado "
                    f"(503, intento {intento}/"
                    f"{MAX_REINTENTOS_TEMPORALES}). "
                    f"Esperando {espera:.1f}s..."
                )

                time.sleep(
                    espera
                )

                continue

            # Cualquier otro error sí se propaga.
            raise

    raise RuntimeError(
        "No fue posible completar la llamada "
        "después de varios reintentos."
    )


def juez_controlado(
    system_prompt,
    user_prompt
):
    return generar_controlado(
        system_prompt,
        user_prompt,
        temperature=0.0,
        model=MODEL_JUEZ
    )


candidatos_ag = [
    c for c in eval_set
    if c.get(
        "chunks_relevantes"
    )
]

if len(candidatos_ag) < 8:
    raise ValueError(
        "Se requieren al menos 8 "
        "casos con referencias."
    )

muestra = random.sample(
    candidatos_ag,
    8
)

faith = []
relev = []
correct = []
detalle_ag = []


for i, caso in enumerate(
    muestra,
    start=1
):
    print(
        f"[{i}/{len(muestra)}] "
        f"{caso['id']} - "
        f"{caso['pregunta'][:70]}"
    )

    tipos = (
        TIPOS_POR_HERRAMIENTA[
            caso[
                "herramienta_esperada"
            ]
        ]
    )

    resultados = recuperar(
        caso["pregunta"],
        tipos=tipos,
        k=K_GENERACION
    )

    contexto = (
        formatear_contexto(
            resultados
        )
    )

    if (
        caso[
            "herramienta_esperada"
        ]
        == "explicar_codigo"
    ):
        # Usa el mismo contexto recuperado para la respuesta.
        system_respuesta = (
            "Eres TutorBot. Explica el código del material académico en español. "
            "Usa únicamente los fragmentos provistos. "
            "Indica el notebook de origen y no inventes código."
        )

        user_respuesta = (
            f"Consulta: {caso['pregunta']}\\n\\n"
            f"Fragmentos de código:\\n{contexto}"
        )

        generada = generar_controlado(
            system_respuesta,
            user_respuesta,
            model=MODEL_CHAT
        )

    elif (
        caso[
            "herramienta_esperada"
        ]
        == "guiar_tarea"
    ):
        system_respuesta = (
            "Eres TutorBot. Entrega una guía numerada paso a paso, "
            "en español, basada únicamente en el material provisto. "
            "Máximo 6 pasos."
        )

        user_respuesta = (
            f"Tarea: {caso['pregunta']}\\n"
            "Contexto del estudiante: material o actividad académica\\n\\n"
            f"Material:\\n{contexto}"
        )

        generada = generar_controlado(
            system_respuesta,
            user_respuesta,
            model=MODEL_CHAT
        )

    else:
        system_respuesta = (
            "Eres TutorBot, asistente académico del diplomado de "
            "Desarrollo de Aplicaciones con IA. "
            "Responde en español usando únicamente la documentación. "
            "No agregues información no sustentada."
        )

        user_respuesta = (
            f"Pregunta: {caso['pregunta']}\\n\\n"
            f"Documentación:\\n{contexto}"
        )

        generada = generar_controlado(
            system_respuesta,
            user_respuesta,
            model=MODEL_CHAT
        )

    f = parsear_total(
        juez_controlado(
            SYS_FAITHFULNESS,
            (
                f"Pregunta: {caso['pregunta']}\\n\\n"
                f"Respuesta generada:\\n{generada}\\n\\n"
                f"Documentación:\\n{contexto}"
            )
        )
    )

    r = parsear_total(
        juez_controlado(
            SYS_RELEVANCY,
            (
                f"Pregunta: {caso['pregunta']}\\n\\n"
                f"Respuesta generada:\\n{generada}"
            )
        )
    )

    c = parsear_total(
        juez_controlado(
            SYS_CORRECTNESS,
            (
                f"Pregunta: {caso['pregunta']}\\n\\n"
                f"Respuesta generada:\\n{generada}\\n\\n"
                f"Respuesta correcta:\\n"
                f"{caso['respuesta_correcta']}"
            )
        )
    )

    if f is not None:
        faith.append(f)

    if r is not None:
        relev.append(r)

    if c is not None:
        correct.append(c)

    chunks_recuperados = [
        d.metadata.get(
            "chunk_id"
        )
        for d, _ in resultados
    ]

    distancias = [
        round(
            float(distancia),
            6
        )
        for _, distancia in resultados
    ]

    detalle_ag.append({
        "id":
            caso["id"],

        "categoria":
            caso.get(
                "categoria"
            ),

        "herramienta":
            caso[
                "herramienta_esperada"
            ],

        "pregunta":
            caso["pregunta"],

        "respuesta_generada":
            generada,

        "respuesta_correcta":
            caso[
                "respuesta_correcta"
            ],

        "contexto_recuperado":
            contexto,

        "chunks_recuperados":
            " | ".join(
                chunks_recuperados
            ),

        "distancias":
            " | ".join(
                map(
                    str,
                    distancias
                )
            ),

        "faithfulness":
            f,

        "answer_relevancy":
            r,

        "answer_correctness":
            c,
    })

    print(
        f"  Faithfulness={f} | "
        f"Relevancy={r} | "
        f"Correctness={c}\\n"
    )


faithfulness_ag = (
    float(np.mean(faith))
    if faith
    else np.nan
)

answer_relevancy_ag = (
    float(np.mean(relev))
    if relev
    else np.nan
)

answer_correctness_ag = (
    float(np.mean(correct))
    if correct
    else np.nan
)


print(
    f"Faithfulness:       "
    f"{faithfulness_ag:.3f}"
)

print(
    f"Answer Relevancy:   "
    f"{answer_relevancy_ag:.3f}"
)

print(
    f"Answer Correctness: "
    f"{answer_correctness_ag:.3f}"
)


detalle_ag_df = pd.DataFrame(
    detalle_ag
)

DETALLE_AG_PATH = (
    RESULTS_DIR
    / "resultados_generacion_detalle.csv"
)

detalle_ag_df.to_csv(
    DETALLE_AG_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "\\nDetalle guardado en:",
    DETALLE_AG_PATH
)


[1/8] EV13 - ¿Qué hace preprocess_text en el notebook de modelado de tópicos?
  Gemini temporalmente saturado (503, intento 1/6). Esperando 12.8s...
  Faithfulness=1.0 | Relevancy=1.0 | Correctness=1.0\n
[2/8] EV25 - Quiero implementar RAG en mi chatbot
  Faithfulness=1.0 | Relevancy=1.0 | Correctness=0.3333333333333333\n
[3/8] EV14 - ¿Qué hace la función convolucion del notebook de filtrado?
  Faithfulness=1.0 | Relevancy=1.0 | Correctness=1.0\n
[4/8] EV02 - ¿Para qué se utiliza Pandas al trabajar con datos tabulares?
  Faithfulness=1.0 | Relevancy=1.0 | Correctness=1.0\n
[5/8] EV09 - ¿Qué caracteriza a una serie temporal?
  Faithfulness=1.0 | Relevancy=1.0 | Correctness=1.0\n
[6/8] EV17 - ¿Qué hace la función PSO del notebook PSO_ABC?
  Faithfulness=1.0 | Relevancy=1.0 | Correctness=1.0\n
[7/8] EV16 - ¿Qué hace la función predict_intent?
  Gemini temporalmente saturado (503, intento 1/6). Esperando 12.5s...
  Faithfulness=1.0 | Relevancy=1.0 | Correctness=1.0\n
[8/8] EV27 - Quiero se

## 15. Resumen de resultados

La tabla reúne las métricas de recuperación, selección de herramientas y generación. Los archivos
de resultados se guardan en la carpeta `resultados` de Google Drive.


In [28]:
import pandas as pd
import numpy as np


def media_segura(
    valores
):
    try:
        if (
            valores is None
            or len(valores) == 0
        ):
            return np.nan

        return float(
            np.mean(valores)
        )

    except Exception:
        return np.nan


routing_res = (
    float(accuracy_ruta)
    if "accuracy_ruta" in globals()
    else media_segura(
        globals().get(
            "aciertos_ruta"
        )
    )
)

faith_res = (
    float(faithfulness_ag)
    if "faithfulness_ag" in globals()
    else np.nan
)

relev_res = (
    float(answer_relevancy_ag)
    if "answer_relevancy_ag" in globals()
    else np.nan
)

correct_res = (
    float(answer_correctness_ag)
    if "answer_correctness_ag" in globals()
    else np.nan
)


resumen = pd.DataFrame([
    (
        "Retriever",
        f"Hit Rate@{K_EVALUACION}",
        media_segura(hit),
        0.80
    ),
    (
        "Retriever",
        "MRR",
        media_segura(rr),
        0.70
    ),
    (
        "Retriever",
        f"Precision@{K_EVALUACION}",
        media_segura(prec),
        None
    ),
    (
        "Retriever",
        f"Recall@{K_EVALUACION}",
        media_segura(rec),
        None
    ),
    (
        "Enrutamiento",
        "Accuracy herramienta",
        routing_res,
        0.85
    ),
    (
        "Generación",
        "Faithfulness",
        faith_res,
        0.90
    ),
    (
        "Generación",
        "Answer Relevancy",
        relev_res,
        0.85
    ),
    (
        "Generación",
        "Answer Correctness",
        correct_res,
        None
    ),
], columns=[
    "Componente",
    "Métrica",
    "Obtenido",
    "Objetivo"
])


def estado_cumplimiento(
    fila
):
    obtenido = fila[
        "Obtenido"
    ]

    objetivo = fila[
        "Objetivo"
    ]

    if pd.isna(
        obtenido
    ):
        return "No evaluado"

    if pd.isna(
        objetivo
    ):
        return "-"

    return (
        "Sí"
        if obtenido >= objetivo
        else "No"
    )


resumen[
    "Cumple"
] = resumen.apply(
    estado_cumplimiento,
    axis=1
)

resumen[
    "Obtenido"
] = resumen[
    "Obtenido"
].round(3)

display(
    resumen
)


RESULTADOS_PATH = (
    RESULTS_DIR
    / "resultados_metricas.csv"
)

resumen.to_csv(
    RESULTADOS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nResultados guardados en:",
    RESULTADOS_PATH
)


,Componente,Métrica,Obtenido,Objetivo,Cumple
0,Retriever,Hit Rate@3,0.444,0.80,No
1,Retriever,MRR,0.315,0.70,No
2,Retriever,Precision@3,0.185,NaN,-
3,Retriever,Recall@3,0.185,NaN,-
4,Enrutamiento,Accuracy herramienta,1.000,0.85,Sí
5,Generación,Faithfulness,1.000,0.90,Sí
6,Generación,Answer Relevancy,1.000,0.85,Sí
7,Generación,Answer Correctness,0.917,NaN,-



Resultados guardados en: /content/Utils_Chatbot_extraido/Utils_Chatbot/resultados/resultados_metricas.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 16. Exportación de archivos

Los archivos generados durante la ejecución se encuentran en las carpetas locales `datos` y
`resultados`. La siguiente celda reúne ambas carpetas en un ZIP para poder descargarlo.


In [29]:
import shutil
from pathlib import Path
from google.colab import files

EXPORT_DIR = Path(
    "/content/TutorBot_export"
)

if EXPORT_DIR.exists():
    shutil.rmtree(
        EXPORT_DIR
    )

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


if DATA_DIR.exists():
    shutil.copytree(
        DATA_DIR,
        EXPORT_DIR / "datos",
        dirs_exist_ok=True
    )


if RESULTS_DIR.exists():
    shutil.copytree(
        RESULTS_DIR,
        EXPORT_DIR / "resultados",
        dirs_exist_ok=True
    )


ZIP_PATH = shutil.make_archive(
    "/content/TutorBot_resultados",
    "zip",
    EXPORT_DIR
)

print(
    "ZIP preparado:",
    ZIP_PATH
)

# Descomenta esta línea cuando quieras descargarlo.
# files.download(ZIP_PATH)


ZIP preparado: /content/TutorBot_resultados.zip


## 17. Preparación del despliegue

Esta sección crea una carpeta autocontenida con la aplicación Gradio, los índices FAISS,
la configuración, las dependencias y el Dockerfile. La API key no se guarda en los archivos:
Railway la recibe mediante la variable de entorno `GEMINI_API_KEY`.


In [30]:
import json
import shutil
from pathlib import Path


RAILWAY_DIR = Path(
    "/content/TutorBot_Railway"
)

if RAILWAY_DIR.exists():
    shutil.rmtree(
        RAILWAY_DIR
    )

RAILWAY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INDICES_RAILWAY_DIR = (
    RAILWAY_DIR
    / "indices"
)

INDICES_RAILWAY_DIR.mkdir(
    parents=True,
    exist_ok=True
)


for nombre, indice in indices.items():
    indice.save_local(
        str(
            INDICES_RAILWAY_DIR
            / nombre
        )
    )


deploy_config = {
    "model_chat":
        MODEL_CHAT,

    "model_routing":
        MODEL_ROUTING,

    "embedding_model":
        "BAAI/bge-m3",

    "k_generacion":
        int(
            K_GENERACION
        ),

    "umbral_distancia":
        float(
            UMBRAL_DISTANCIA
        ),
}


with open(
    RAILWAY_DIR
    / "deploy_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        deploy_config,
        f,
        ensure_ascii=False,
        indent=2
    )


APP_PY = 'import os\nimport asyncio\nfrom pathlib import Path\n\nimport gradio as gr\nimport torch\nimport torch.nn.functional as F\n\nfrom transformers import AutoTokenizer, AutoModel\nfrom langchain_core.embeddings.embeddings import Embeddings\nfrom langchain_community.vectorstores import FAISS\nfrom langchain_community.vectorstores.utils import DistanceStrategy\n\nfrom google import genai\nfrom google.genai.types import GenerateContentConfig\n\nfrom fastmcp import FastMCP, Client\n\nimport json\n\n\nBASE_DIR = Path(__file__).resolve().parent\nCONFIG_PATH = BASE_DIR / "deploy_config.json"\nINDICES_DIR = BASE_DIR / "indices"\n\n\nwith open(\n    CONFIG_PATH,\n    "r",\n    encoding="utf-8"\n) as f:\n    DEPLOY_CONFIG = json.load(f)\n\n\nMODEL_CHAT = os.getenv(\n    "MODEL_CHAT",\n    DEPLOY_CONFIG[\n        "model_chat"\n    ]\n)\n\nMODEL_ROUTING = os.getenv(\n    "MODEL_ROUTING",\n    DEPLOY_CONFIG[\n        "model_routing"\n    ]\n)\n\nK_GENERACION = int(\n    DEPLOY_CONFIG[\n        "k_generacion"\n    ]\n)\n\nUMBRAL_DISTANCIA = float(\n    DEPLOY_CONFIG[\n        "umbral_distancia"\n    ]\n)\n\n\napi_key = os.getenv(\n    "GEMINI_API_KEY"\n)\n\nif not api_key:\n    raise RuntimeError(\n        "Falta la variable de entorno GEMINI_API_KEY."\n    )\n\n\ngemini_client = genai.Client(\n    api_key=api_key\n)\n\n\nclass SentenceEmbeddings(Embeddings):\n    def __init__(\n        self,\n        model_name,\n        batch_size=8\n    ):\n        self.device = torch.device(\n            "cuda"\n            if torch.cuda.is_available()\n            else "cpu"\n        )\n\n        self.model_name = model_name\n        self.batch_size = batch_size\n\n        self.tokenizer = (\n            AutoTokenizer.from_pretrained(\n                model_name\n            )\n        )\n\n        self.model = (\n            AutoModel.from_pretrained(\n                model_name\n            )\n            .to(\n                self.device\n            )\n            .eval()\n        )\n\n    def _encode(\n        self,\n        textos\n    ):\n        vectores = []\n\n        for i in range(\n            0,\n            len(textos),\n            self.batch_size\n        ):\n            lote = textos[\n                i:i + self.batch_size\n            ]\n\n            entradas = self.tokenizer(\n                lote,\n                padding=True,\n                truncation=True,\n                max_length=512,\n                return_tensors="pt"\n            ).to(\n                self.device\n            )\n\n            with torch.no_grad():\n                salida = self.model(\n                    **entradas\n                )\n\n            emb = F.normalize(\n                salida.last_hidden_state[\n                    :,\n                    0\n                ],\n                p=2,\n                dim=1\n            )\n\n            vectores.extend(\n                emb.cpu().tolist()\n            )\n\n        return vectores\n\n    def embed_documents(\n        self,\n        texts\n    ):\n        return self._encode(\n            list(\n                texts\n            )\n        )\n\n    def embed_query(\n        self,\n        text\n    ):\n        return self._encode(\n            [\n                text\n            ]\n        )[0]\n\n\nsentence_embeddings = (\n    SentenceEmbeddings(\n        DEPLOY_CONFIG.get(\n            "embedding_model",\n            "BAAI/bge-m3"\n        )\n    )\n)\n\n\nINDICE_POR_NOMBRE = {}\n\nfor nombre in (\n    "concepto",\n    "codigo",\n    "tarea",\n    "todos"\n):\n    ruta = (\n        INDICES_DIR\n        / nombre\n    )\n\n    INDICE_POR_NOMBRE[\n        nombre\n    ] = FAISS.load_local(\n        str(\n            ruta\n        ),\n        sentence_embeddings,\n        allow_dangerous_deserialization=True,\n        distance_strategy=(\n            DistanceStrategy.COSINE\n        )\n    )\n\n\nMAPA_INDICE = {\n    frozenset({\n        "slide",\n        "transcripcion"\n    }): "concepto",\n\n    frozenset({\n        "codigo"\n    }): "codigo",\n\n    frozenset({\n        "slide",\n        "transcripcion",\n        "codigo",\n        "administrativo"\n    }): "tarea",\n}\n\n\ndef obtener_indice(\n    tipos=None\n):\n    if not tipos:\n        return INDICE_POR_NOMBRE[\n            "todos"\n        ]\n\n    clave = frozenset(\n        tipos\n    )\n\n    if clave not in MAPA_INDICE:\n        raise ValueError(\n            "Tipos de contenido no configurados."\n        )\n\n    return INDICE_POR_NOMBRE[\n        MAPA_INDICE[\n            clave\n        ]\n    ]\n\n\ndef recuperar(\n    consulta,\n    tipos=None,\n    k=K_GENERACION\n):\n    return obtener_indice(\n        tipos\n    ).similarity_search_with_score(\n        consulta,\n        k=k\n    )\n\n\ndef formatear_contexto(resultados):\n    partes = []\n\n    for doc, distancia in resultados:\n        meta = doc.metadata\n\n        origen = (\n            f"Ciclo {meta.get(\'ciclo\')} · "\n            f"{meta.get(\'curso\', \'Curso no identificado\')} · "\n            f"{meta.get(\'fuente\', \'Material académico\')}"\n        )\n\n        if meta.get("clase") is not None:\n            origen += f" · clase {meta[\'clase\']}"\n\n        partes.append(\n            f"[Fuente: {origen} | distancia {distancia:.3f}]\\n"\n            f"{doc.page_content}"\n        )\n\n    return "\\n\\n".join(partes)\n\n\n\ndef _generar(\n    system_prompt,\n    user_prompt,\n    temperature=0.2,\n    model=None\n):\n    modelo = (\n        model\n        or MODEL_CHAT\n    )\n\n    config_kwargs = {\n        "system_instruction":\n            system_prompt\n    }\n\n    if not modelo.startswith(\n        "gemini-3"\n    ):\n        config_kwargs[\n            "temperature"\n        ] = temperature\n\n    config = GenerateContentConfig(\n        **config_kwargs\n    )\n\n    mensajes = [\n        genai.types.Content(\n            role="user",\n            parts=[\n                genai.types.Part.from_text(\n                    text=user_prompt\n                )\n            ]\n        )\n    ]\n\n    return (\n        gemini_client.models\n        .generate_content(\n            model=modelo,\n            contents=mensajes,\n            config=config\n        )\n        .text\n    )\n\n\nETIQUETA_FALLBACK = (\n    "[Fallback]"\n)\n\n\ndef fn_fallback(\n    consulta,\n    motivo="fuera_alcance"\n):\n    if motivo == "fuera_alcance":\n        return (\n            f"{ETIQUETA_FALLBACK} "\n            "No encontré evidencia suficiente en el material "\n            "del curso para responder esa consulta. "\n            "Puedo ayudarte con conceptos, código, notebooks "\n            "y entregables del curso."\n        )\n\n    if motivo == "sin_codigo":\n        return (\n            f"{ETIQUETA_FALLBACK} "\n            "No encontré fragmentos de código relacionados. "\n            "Indícame el notebook, función o fragmento."\n        )\n\n    if motivo == "sin_guia":\n        return (\n            f"{ETIQUETA_FALLBACK} "\n            "No encontré material suficiente para construir "\n            "una guía sobre esa tarea."\n        )\n\n    if motivo == "mensaje_vacio":\n        return (\n            f"{ETIQUETA_FALLBACK} "\n            "Escribe una consulta para poder ayudarte."\n        )\n\n    return (\n        f"{ETIQUETA_FALLBACK} "\n        "No pude completar la consulta en este momento. "\n        "Intenta nuevamente o reformula la pregunta."\n    )\n\n\ndef fn_responder_concepto(\n    pregunta\n):\n    resultados = recuperar(\n        pregunta,\n        tipos={\n            "slide",\n            "transcripcion"\n        },\n        k=K_GENERACION\n    )\n\n    suficiente = (\n        bool(\n            resultados\n        )\n        and resultados[0][1]\n        <= UMBRAL_DISTANCIA\n    )\n\n    if not suficiente:\n        return fn_fallback(\n            pregunta,\n            "fuera_alcance"\n        )\n\n    return _generar(\n        (\n            "Eres TutorBot, asistente del curso de Diseño "\n            "de Chatbots Conversacionales. Responde en español "\n            "usando únicamente la documentación proporcionada. "\n            "No agregues información no sustentada."\n        ),\n        (\n            f"Pregunta: {pregunta}\\n\\n"\n            f"Documentación:\\n"\n            f"{formatear_contexto(resultados)}"\n        )\n    )\n\n\ndef fn_explicar_codigo(\n    pregunta\n):\n    resultados = recuperar(\n        pregunta,\n        tipos={"codigo"},\n        k=K_GENERACION\n    )\n\n    if not resultados:\n        return fn_fallback(\n            pregunta,\n            "sin_codigo"\n        )\n\n    return _generar(\n        (\n            "Eres TutorBot. Explica el código del material académico "\n            "en español usando únicamente los fragmentos "\n            "proporcionados. Indica el notebook de origen."\n        ),\n        (\n            f"Consulta: {pregunta}\\n\\n"\n            f"Fragmentos:\\n"\n            f"{formatear_contexto(resultados)}"\n        )\n    )\n\n\ndef fn_guiar_tarea(\n    tarea,\n    detalle=""\n):\n    if not detalle.strip():\n        return (\n            f"Para guiarte con \'{tarea}\' necesito un dato más: "\n            "¿sobre qué clase, notebook o entregable "\n            "específico lo necesitas?"\n        )\n\n    consulta = (\n        f"{tarea} {detalle}"\n    )\n\n    resultados = recuperar(\n        consulta,\n        tipos={\n            "slide",\n            "transcripcion",\n            "codigo",\n            "administrativo"\n        },\n        k=K_GENERACION\n    )\n\n    if not resultados:\n        return fn_fallback(\n            consulta,\n            "sin_guia"\n        )\n\n    return _generar(\n        (\n            "Eres TutorBot. Entrega una guía numerada "\n            "paso a paso, en español, basada únicamente "\n            "en el material provisto. Máximo 6 pasos."\n        ),\n        (\n            f"Tarea: {tarea}\\n"\n            f"Contexto: {detalle}\\n\\n"\n            f"Material:\\n"\n            f"{formatear_contexto(resultados)}"\n        )\n    )\n\n\ndef fn_pedir_aclaracion(\n    consulta\n):\n    return _generar(\n        (\n            "Eres TutorBot. La consulta del estudiante es "\n            "ambigua. Devuelve una sola pregunta corta "\n            "en español para desambiguarla."\n        ),\n        f"Consulta ambigua: {consulta}",\n        temperature=0.3\n    )\n\n\nmcp_server = FastMCP(\n    "tutorbot"\n)\n\n\n@mcp_server.tool()\ndef responder_concepto(\n    pregunta: str\n) -> str:\n    """Responde teoría, conceptos y preguntas claras de conocimiento."""\n    return fn_responder_concepto(\n        pregunta\n    )\n\n\n@mcp_server.tool()\ndef explicar_codigo(\n    pregunta: str\n) -> str:\n    """Explica código existente en los notebooks del curso."""\n    return fn_explicar_codigo(\n        pregunta\n    )\n\n\n@mcp_server.tool()\ndef guiar_tarea(\n    tarea: str,\n    detalle: str = ""\n) -> str:\n    """Guía tareas de creación, modificación, implementación o despliegue."""\n    return fn_guiar_tarea(\n        tarea,\n        detalle\n    )\n\n\n@mcp_server.tool()\ndef pedir_aclaracion(\n    consulta: str\n) -> str:\n    """Pide aclaración cuando la consulta es ambigua o incompleta."""\n    return fn_pedir_aclaracion(\n        consulta\n    )\n\n\nmcp_client = Client(\n    mcp_server\n)\n\n\nSYSTEM_AGENTE = """\nEres TutorBot, asistente académico del diplomado de Desarrollo\nde Aplicaciones con IA.\n\nSelecciona siempre una herramienta.\n\nMEMORIA DE CONVERSACIÓN:\n\n- Antes de seleccionar una herramienta, revisa el historial reciente.\n- Resuelve referencias como "eso", "ese", "esa", "cada uno",\n  "el anterior", "el modelo", "la función", "cómo lo hago",\n  "dame ejemplos" o expresiones similares usando el historial.\n- Si la consulta puede entenderse usando el historial,\n  NO uses pedir_aclaracion.\n- Al invocar una herramienta, convierte la consulta en una\n  pregunta autosuficiente cuando sea necesario.\n\nEjemplo:\n\nHistorial:\nUsuario: ¿Qué diferencia hay entre aprendizaje supervisado\ny no supervisado?\nTutorBot: ...\n\nNueva consulta:\nDame ejemplos de cada uno\n\nLa herramienta debe recibir algo equivalente a:\n"Dame ejemplos de aprendizaje supervisado y aprendizaje no supervisado."\n\nReglas de selección:\n\n1. responder_concepto\n   - teoría, definiciones y conceptos;\n   - preguntas claras de conocimiento;\n   - preguntas de seguimiento sobre conceptos mencionados\n     anteriormente en la conversación;\n   - el control de alcance se realiza dentro de la herramienta.\n\n2. explicar_codigo\n   - funciones, clases, librerías, celdas o fragmentos\n     existentes en los notebooks;\n   - preguntas de seguimiento sobre código mencionado\n     anteriormente;\n   - una pregunta que empieza por "cómo" sigue siendo de código\n     si pregunta cómo funciona, se define, se configura o se usa\n     código existente.\n\n3. guiar_tarea\n   - el estudiante expresa intención de crear, modificar,\n     implementar, desplegar o completar algo;\n   - preguntas sobre requisitos de una actividad o entregable;\n   - preguntas de seguimiento sobre una tarea previamente\n     mencionada.\n\n4. pedir_aclaracion\n   - úsala solamente si la consulta sigue siendo ambigua\n     después de revisar el historial;\n   - no la uses si el referente puede deducirse de mensajes\n     anteriores;\n   - no la uses para una pregunta clara solo porque esté\n     fuera del alcance del corpus.\n\nEl fallback se activa automáticamente cuando no existe evidencia\nsuficiente en el corpus o cuando la interacción no puede completarse.\n\nDevuelve la respuesta de la herramienta en español.\nSi contiene la etiqueta [Fallback], consérvala.\n"""\n\ndef extraer_texto_gradio(\n    content\n):\n    """\n    Convierte el contenido de mensajes de Gradio\n    a texto simple.\n    """\n\n    if isinstance(\n        content,\n        str\n    ):\n        return content.strip()\n\n    if isinstance(\n        content,\n        list\n    ):\n        partes = []\n\n        for bloque in content:\n\n            if isinstance(\n                bloque,\n                str\n            ):\n                partes.append(\n                    bloque\n                )\n\n            elif isinstance(\n                bloque,\n                dict\n            ):\n                if (\n                    bloque.get("type")\n                    == "text"\n                ):\n                    texto = bloque.get(\n                        "text",\n                        ""\n                    )\n\n                    if texto:\n                        partes.append(\n                            str(texto)\n                        )\n\n        return "\\n".join(\n            partes\n        ).strip()\n\n    if isinstance(\n        content,\n        dict\n    ):\n        if (\n            content.get("type")\n            == "text"\n        ):\n            return str(\n                content.get(\n                    "text",\n                    ""\n                )\n            ).strip()\n\n    return ""\n\ndef normalizar_historial_gradio(\n    historial,\n    max_mensajes=8\n):\n    """\n    Convierte el historial de Gradio 6\n    al formato simple que utiliza TutorBot.\n    """\n\n    historial = (\n        historial\n        or []\n    )[-max_mensajes:]\n\n    normalizado = []\n\n    for mensaje in historial:\n\n        if not isinstance(\n            mensaje,\n            dict\n        ):\n            continue\n\n        role = mensaje.get(\n            "role",\n            ""\n        )\n\n        if role not in {\n            "user",\n            "assistant",\n            "model"\n        }:\n            continue\n\n        content = extraer_texto_gradio(\n            mensaje.get(\n                "content",\n                ""\n            )\n        )\n\n        if not content:\n            continue\n\n        normalizado.append({\n            "role":\n                role,\n\n            "content":\n                content,\n        })\n\n    return normalizado\n\ndef construir_mensajes(\n    consulta,\n    historial=None,\n    max_mensajes=8\n):\n    mensajes = []\n\n    historial = (\n        historial\n        or []\n    )[-max_mensajes:]\n\n    for turno in historial:\n        role = turno.get(\n            "role",\n            "user"\n        )\n\n        contenido = turno.get(\n            "content",\n            ""\n        )\n\n        if role == "assistant":\n            role = "model"\n\n        if role not in {\n            "user",\n            "model"\n        }:\n            continue\n\n        if not isinstance(\n            contenido,\n            str\n        ):\n            continue\n\n        if not contenido.strip():\n            continue\n\n        mensajes.append(\n            genai.types.Content(\n                role=role,\n                parts=[\n                    genai.types.Part.from_text(\n                        text=contenido\n                    )\n                ]\n            )\n        )\n\n    mensajes.append(\n        genai.types.Content(\n            role="user",\n            parts=[\n                genai.types.Part.from_text(\n                    text=consulta\n                )\n            ]\n        )\n    )\n\n    return mensajes\n\n\nasync def agente(\n    consulta,\n    historial=None,\n    max_iterations=3\n):\n    consulta = (\n        consulta\n        or ""\n    ).strip()\n\n    if not consulta:\n        return (\n            fn_fallback(\n                "",\n                "mensaje_vacio"\n            ),\n            ["fallback"]\n        )\n\n    for intento in range(\n        1,\n        max_iterations + 1\n    ):\n        try:\n            async with mcp_client:\n                config = GenerateContentConfig(\n                    system_instruction=(\n                        SYSTEM_AGENTE\n                    ),\n                    tools=[\n                        mcp_client.session\n                    ],\n                )\n\n                respuesta = (\n                    await gemini_client.aio.models\n                    .generate_content(\n                        model=MODEL_ROUTING,\n                        contents=construir_mensajes(\n                            consulta,\n                            historial\n                        ),\n                        config=config\n                    )\n                )\n\n            usadas = []\n\n            for contenido in (\n                respuesta.automatic_function_calling_history\n                or []\n            ):\n                for parte in (\n                    contenido.parts\n                    or []\n                ):\n                    if getattr(\n                        parte,\n                        "function_call",\n                        None\n                    ):\n                        usadas.append(\n                            parte.function_call.name\n                        )\n\n            texto = (\n                respuesta.text\n                or ""\n            ).strip()\n\n            if not texto:\n                raise ValueError(\n                    "respuesta vacía"\n                )\n\n            return (\n                texto,\n                usadas\n            )\n\n        except Exception as e:\n            if intento == max_iterations:\n                return (\n                    fn_fallback(\n                        consulta,\n                        "error_temporal"\n                    ),\n                    ["fallback"]\n                )\n\n            mensaje = str(\n                e\n            )\n\n            if (\n                "429" in mensaje\n                or "RESOURCE_EXHAUSTED" in mensaje\n                or "503" in mensaje\n                or "UNAVAILABLE" in mensaje\n            ):\n                espera = min(\n                    10 * intento,\n                    60\n                )\n            else:\n                espera = (\n                    1.5 * intento\n                )\n\n            await asyncio.sleep(\n                espera\n            )\n\ndef historial_a_texto(\n    historial,\n    max_mensajes=6\n):\n    historial = (\n        historial\n        or []\n    )[-max_mensajes:]\n\n    partes = []\n\n    for mensaje in historial:\n\n        role = mensaje.get(\n            "role",\n            ""\n        )\n\n        content = mensaje.get(\n            "content",\n            ""\n        )\n\n        if not content:\n            continue\n\n        if role == "user":\n            etiqueta = "Usuario"\n\n        elif role in {\n            "assistant",\n            "model"\n        }:\n            etiqueta = "TutorBot"\n\n        else:\n            continue\n\n        partes.append(\n            f"{etiqueta}: {content}"\n        )\n\n    return "\\n".join(\n        partes\n    )\n\n\nasync def contextualizar_consulta(\n    consulta,\n    historial\n):\n    """\n    Convierte una pregunta de seguimiento en una consulta\n    autosuficiente antes de enviarla al router y al RAG.\n    """\n\n    if not historial:\n        return consulta\n\n    historial_texto = historial_a_texto(\n        historial\n    )\n\n    if not historial_texto:\n        return consulta\n\n    system = """\nEres un componente de contextualización de consultas\npara un asistente académico.\n\nTu única tarea es reescribir la CONSULTA ACTUAL como una\nconsulta autosuficiente utilizando el HISTORIAL.\n\nReglas:\n\n- NO respondas la pregunta.\n- Devuelve únicamente la consulta reescrita.\n- Resuelve referencias como:\n  "eso", "ese", "esa", "ellos", "cada uno",\n  "el anterior", "la anterior", "ese modelo",\n  "esa función", "dame ejemplos", "¿y cómo?",\n  "¿y por qué?", "¿y cuál es su fórmula?"\n  usando el historial.\n- Conserva la intención original del usuario.\n- No inventes información que no aparezca en el historial.\n- Si la consulta ya es autosuficiente, devuélvela sin cambios.\n"""\n\n    user = (\n        f"HISTORIAL:\\n"\n        f"{historial_texto}\\n\\n"\n        f"CONSULTA ACTUAL:\\n"\n        f"{consulta}"\n    )\n\n    try:\n        respuesta = (\n            await gemini_client.aio.models.generate_content(\n                model=MODEL_ROUTING,\n                contents=[\n                    genai.types.Content(\n                        role="user",\n                        parts=[\n                            genai.types.Part.from_text(\n                                text=user\n                            )\n                        ]\n                    )\n                ],\n                config=GenerateContentConfig(\n                    system_instruction=system\n                )\n            )\n        )\n\n        contextualizada = (\n            respuesta.text\n            or ""\n        ).strip()\n\n        return (\n            contextualizada\n            if contextualizada\n            else consulta\n        )\n\n    except Exception as e:\n        print(\n            "No se pudo contextualizar:",\n            e\n        )\n\n        return consulta\n\nasync def responder_gradio(\n    message,\n    history\n):\n    # 1. Convierte el follow-up en una consulta autosuficiente\n    consulta_contextual = (\n        await contextualizar_consulta(\n            message,\n            history\n        )\n    )\n\n    print(\n        "Consulta original:",\n        message\n    )\n\n    print(\n        "Consulta contextualizada:",\n        consulta_contextual\n    )\n\n    # 2. El agente recibe la consulta ya resuelta\n    respuesta, _ = await agente(\n        consulta_contextual,\n        historial=history\n    )\n\n    return respuesta\n\n\ndemo = gr.ChatInterface(\n    fn=responder_gradio,\n    title="TutorBot",\n    description=(\n        "Asistente académico del curso de "\n        "Diseño de Chatbots Conversacionales"\n    ),\n    examples=[\n        "¿Qué diferencia hay entre aprendizaje supervisado y no supervisado?",\n        "¿Qué hace la clase QLearningAgent?",\n        "Quiero crear un dashboard con Streamlit",\n        "¿Qué es una CNN?",\n        "Quiero implementar RAG en mi chatbot",\n        "¿Qué caracteriza a una serie temporal?",\n    ],\n    save_history=True,\n)\n\n\nif __name__ == "__main__":\n    port = int(\n        os.getenv(\n            "PORT",\n            "7860"\n        )\n    )\n\n    demo.queue()\n\n    demo.launch(\n        server_name="0.0.0.0",\n        server_port=port,\n        show_error=True\n    )\n'


with open(
    RAILWAY_DIR
    / "app.py",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        APP_PY
    )


REQUIREMENTS = """google-genai==1.56.0
fastmcp==3.1.1
gradio==6.24.0
langchain-community==0.4.1
langchain-text-splitters==1.1.1
faiss-cpu==1.13.2
transformers
torch
"""


with open(
    RAILWAY_DIR
    / "requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        REQUIREMENTS
    )


DOCKERFILE = """FROM python:3.11-slim

ENV PYTHONUNBUFFERED=1
ENV PIP_NO_CACHE_DIR=1
ENV HF_HOME=/app/.cache/huggingface

WORKDIR /app

RUN apt-get update \\
    && apt-get install -y --no-install-recommends git libgomp1 \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .

RUN pip install --upgrade pip \\
    && pip install -r requirements.txt

COPY . .

CMD ["python", "app.py"]
"""


with open(
    RAILWAY_DIR
    / "Dockerfile",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        DOCKERFILE
    )


RAILWAY_JSON = {
    "$schema":
        "https://railway.com/railway.schema.json",

    "build": {
        "builder":
            "DOCKERFILE"
    }
}


with open(
    RAILWAY_DIR
    / "railway.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        RAILWAY_JSON,
        f,
        indent=2
    )


GITIGNORE = """.env
__pycache__/
*.pyc
.ipynb_checkpoints/
"""


with open(
    RAILWAY_DIR
    / ".gitignore",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        GITIGNORE
    )


ZIP_RAILWAY = shutil.make_archive(
    "/content/TutorBot_Railway",
    "zip",
    RAILWAY_DIR
)


print(
    "Proyecto preparado en:",
    RAILWAY_DIR
)

print(
    "ZIP preparado en:",
    ZIP_RAILWAY
)

print(
    "\nArchivos:"
)

for ruta in sorted(
    RAILWAY_DIR.rglob("*")
):
    if ruta.is_file():
        print(
            " -",
            ruta.relative_to(
                RAILWAY_DIR
            )
        )


# Para descargar el proyecto:
# from google.colab import files
# files.download(ZIP_RAILWAY)


Proyecto preparado en: /content/TutorBot_Railway
ZIP preparado en: /content/TutorBot_Railway.zip

Archivos:
 - .gitignore
 - Dockerfile
 - app.py
 - deploy_config.json
 - indices/codigo/index.faiss
 - indices/codigo/index.pkl
 - indices/concepto/index.faiss
 - indices/concepto/index.pkl
 - indices/tarea/index.faiss
 - indices/tarea/index.pkl
 - indices/todos/index.faiss
 - indices/todos/index.pkl
 - railway.json
 - requirements.txt


In [31]:
from google.colab import files

files.download(
    "/content/TutorBot_Railway.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>